# Predator: predicting online purchases and explaining why

**Question:** will an online shopping session end in a purchase, and which behaviours drive that prediction?

We use the **Online Shoppers Purchasing Intention** dataset (UCI / Kaggle): 12,330 browsing sessions from an online store, each described by 17 features (pages visited, time spent, Google Analytics metrics such as bounce rate, exit rate and page value, plus month, visitor type and technical details). The target `Revenue` says whether the session ended in a purchase; only about 15% did.

**Models**
- Main models: **Decision Tree** and **K-Nearest Neighbours** (KNN, trained with and without undersampling)
- Baseline: **Gaussian Naive Bayes**
- Benchmark beyond the main models: **XGBoost**

**What this notebook covers:** data checks and EDA, a leak-free preprocessing pipeline, tuning with 5-fold stratified cross-validation, test-set evaluation with confidence intervals and significance tests, learning and validation curves, feature importance and partial dependence, an ablation without `PageValues` (and whether it leaks the answer), a profit-based decision threshold with a sensitivity analysis, probability calibration, and an error analysis of the sessions the best model gets wrong.

> **Full project on GitHub:** [https://github.com/pxnz3r/Predator](https://github.com/pxnz3r/Predator) (the `predator` package, tests, documentation and a **Streamlit demo app**). This notebook is a self-contained version: all the code is included below.

In [ ]:
import os
import warnings

import pandas as pd
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def show(path, width=900):
    """Display a saved figure inside the notebook."""
    display(Image(filename=path, width=width))

## 1. Loading and checking the data

`find_dataset()` looks in a local `data/` folder first and then searches `/kaggle/input/`, so the same code runs on a laptop and on Kaggle. We then check for missing values, duplicates and negative numbers.

<details><summary>The code below is <code>predator/data.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
data.py - Loading, checking and exploring the dataset.

This file has three jobs:
1. Find the CSV file, both on a laptop (data/ folder) and on Kaggle (/kaggle/input/).
2. Load it and run simple data quality checks (missing values, duplicates, types).
3. Draw the exploratory data analysis (EDA) plots and split the data into
   a training set and a test set.
"""

import os

import matplotlib

# "Agg" lets matplotlib save plots to files without opening a window.
# This keeps main.py working on servers and on Kaggle.
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------------
# Settings used across the whole project
# ---------------------------------------------------------------------------

DATASET_FILENAME = "online_shoppers_intention.csv"
LOCAL_DATA_FOLDER = "data"
KAGGLE_INPUT_FOLDER = "/kaggle/input"
FIGURES_FOLDER = os.path.join("outputs", "figures")

# The same random_state everywhere makes every run give the same results.
RANDOM_STATE = 42
TEST_SIZE = 0.20

TARGET_COLUMN = "Revenue"

# Numbers we can measure on a scale (counts, seconds, rates).
NUMERICAL_FEATURES = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
]

# Labels or codes. OperatingSystems = 2 is not "twice" OperatingSystems = 1,
# so these columns are treated as categories even when they hold numbers.
CATEGORICAL_FEATURES = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

# Colours used for the two classes in every plot.
NO_PURCHASE_COLOR = "#9AA5B1"
PURCHASE_COLOR = "#2A9D8F"


def find_dataset(filename=DATASET_FILENAME):
    """
    Look for the dataset file in the places it is likely to be.

    First it checks the local data/ folder (when running on your own computer).
    If it is not there, it searches every folder under /kaggle/input/
    (when running as a Kaggle notebook, where datasets are mounted there).

    Input:
        filename: the name of the CSV file to look for.
    Returns:
        The full path to the file as a string, or None if it was not found.
    """
    # 1. Local copy, for example data/online_shoppers_intention.csv
    local_path = os.path.join(LOCAL_DATA_FOLDER, filename)
    if os.path.isfile(local_path):
        return local_path

    # 2. Kaggle copy. The exact sub-folder depends on which dataset was
    #    attached, so we walk through all folders under /kaggle/input/.
    if os.path.isdir(KAGGLE_INPUT_FOLDER):
        for folder, _subfolders, files in os.walk(KAGGLE_INPUT_FOLDER):
            if filename in files:
                return os.path.join(folder, filename)

    return None


def load_dataset(path=None):
    """
    Load the Online Shoppers dataset into a pandas DataFrame.

    If no path is given, find_dataset() is used to locate the file.
    The target column Revenue is turned from True/False into 1/0 so the
    models and metrics can work with it directly.

    Input:
        path: optional path to the CSV file.
    Returns:
        A pandas DataFrame with one row per shopping session.
    Raises:
        FileNotFoundError with a friendly explanation if the file is missing.
    """
    if path is None:
        path = find_dataset()

    if path is None or not os.path.isfile(path):
        message = (
            "\n\nCould not find the dataset file '" + DATASET_FILENAME + "'.\n\n"
            "How to fix it:\n"
            "  1. Download 'Online Shoppers Purchasing Intention Dataset' from Kaggle\n"
            "     (https://www.kaggle.com/datasets/henrysue/online-shoppers-intention)\n"
            "     or from the UCI Machine Learning Repository\n"
            "     (https://archive.ics.uci.edu/dataset/468).\n"
            "  2. Put the CSV file here: " + os.path.join(LOCAL_DATA_FOLDER, DATASET_FILENAME) + "\n"
            "  3. Run the command again.\n"
        )
        raise FileNotFoundError(message)

    print("Loading dataset from:", path)
    data = pd.read_csv(path)

    # pandas reads TRUE/FALSE as booleans. We keep Weekend as True/False
    # (it is a category) but turn the target into 0 = no purchase, 1 = purchase.
    data[TARGET_COLUMN] = data[TARGET_COLUMN].astype(int)
    data["Weekend"] = data["Weekend"].astype(bool)
    return data


def validate_dataset(data):
    """
    Run basic data quality checks and print what we find.

    Checks: expected columns are present, number of rows, missing values,
    duplicate rows, and the share of sessions that ended in a purchase.

    Input:
        data: the DataFrame returned by load_dataset().
    Returns:
        A dictionary with the results of each check.
    Raises:
        ValueError if a required column is missing, because nothing
        after this point can work without it.
    """
    print("\nChecking data quality...")

    expected_columns = ALL_FEATURES + [TARGET_COLUMN]
    missing_columns = [column for column in expected_columns if column not in data.columns]
    if len(missing_columns) > 0:
        raise ValueError("The dataset is missing these columns: " + ", ".join(missing_columns))

    number_of_rows = len(data)
    missing_values = int(data.isna().sum().sum())
    duplicate_rows = int(data.duplicated().sum())
    purchase_rate = float(data[TARGET_COLUMN].mean())

    print("  Rows (sessions):", number_of_rows)
    print("  Feature columns:", len(ALL_FEATURES))
    print("  Missing values:", missing_values)
    print("  Duplicate rows:", duplicate_rows)
    print(f"  Sessions that ended in a purchase: {purchase_rate:.1%}")

    # We keep duplicate rows. In this dataset they are mostly very short
    # sessions (one page, then leave), which really do repeat in real life.
    # Removing them would make the data less like real traffic.
    if duplicate_rows > 0:
        print("  Note: duplicate rows are kept, they are mostly identical one-page visits.")

    # Negative counts or durations would mean broken data.
    negative_numbers = int((data[NUMERICAL_FEATURES] < 0).sum().sum())
    print("  Negative values in numerical columns:", negative_numbers)

    report = {
        "rows": number_of_rows,
        "missing_values": missing_values,
        "duplicate_rows": duplicate_rows,
        "negative_values": negative_numbers,
        "purchase_rate": purchase_rate,
    }
    return report


def split_data(data):
    """
    Split the data into a training set (80%) and a test set (20%).

    The split is stratified: both sets keep the same share of buyers
    (about 15%). Without this, the small buyer class could end up
    over- or under-represented in the test set by chance.

    The test set is put aside and only used at the very end, so the
    final scores tell us how the models do on sessions they never saw.

    Input:
        data: the full DataFrame.
    Returns:
        train_features, test_features, train_target, test_target
    """
    features = data[ALL_FEATURES]
    target = data[TARGET_COLUMN]

    train_features, test_features, train_target, test_target = train_test_split(
        features,
        target,
        test_size=TEST_SIZE,
        stratify=target,
        random_state=RANDOM_STATE,
    )

    print("\nSplit the data (stratified 80/20):")
    print("  Training sessions:", len(train_features), f"(buyers: {train_target.mean():.1%})")
    print("  Test sessions:    ", len(test_features), f"(buyers: {test_target.mean():.1%})")
    return train_features, test_features, train_target, test_target


# ---------------------------------------------------------------------------
# Exploratory data analysis (EDA) plots
# ---------------------------------------------------------------------------


def save_figure(figure, filename):
    """
    Save a matplotlib figure into outputs/figures/ and close it.

    Closing the figure frees memory, which matters because the pipeline
    makes many plots in a row.

    Input:
        figure: the matplotlib figure to save.
        filename: the file name, for example "class_balance.png".
    Returns:
        The full path of the saved image.
    """
    os.makedirs(FIGURES_FOLDER, exist_ok=True)
    path = os.path.join(FIGURES_FOLDER, filename)
    figure.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(figure)
    return path


def plot_class_balance(data):
    """
    Bar chart of how many sessions did and did not end in a purchase.

    This shows the class imbalance: only about 15% of sessions are buyers.
    A model that always says "no purchase" would already be about 85%
    accurate, which is why accuracy alone is a misleading score here.

    Input:
        data: the full DataFrame.
    Returns:
        The path of the saved plot.
    """
    counts = data[TARGET_COLUMN].value_counts().sort_index()
    labels = ["No purchase", "Purchase"]

    figure, axis = plt.subplots(figsize=(6, 4))
    bars = axis.bar(labels, counts.values, color=[NO_PURCHASE_COLOR, PURCHASE_COLOR])
    for bar, count in zip(bars, counts.values, strict=True):
        share = count / counts.sum()
        axis.text(
            bar.get_x() + bar.get_width() / 2, count, f"{count:,}\n({share:.1%})", ha="center", va="bottom"
        )
    axis.set_title("Class balance: most sessions do not buy")
    axis.set_ylabel("Number of sessions")
    axis.set_ylim(0, counts.max() * 1.2)
    return save_figure(figure, "class_balance.png")


def plot_numerical_distributions(data):
    """
    Histograms of every numerical feature, split by purchase vs no purchase.

    Most of these features are heavily skewed (many zeros, a few very large
    values), so the x-axis is cut at the 99th percentile to keep the shape
    readable.

    Input:
        data: the full DataFrame.
    Returns:
        The path of the saved plot.
    """
    figure, axes = plt.subplots(2, 5, figsize=(20, 7))
    axes = axes.flatten()

    for index, column in enumerate(NUMERICAL_FEATURES):
        axis = axes[index]
        upper_limit = data[column].quantile(0.99)
        clipped = data[data[column] <= upper_limit]
        sns.histplot(
            data=clipped,
            x=column,
            hue=TARGET_COLUMN,
            bins=30,
            stat="density",
            common_norm=False,
            element="step",
            ax=axis,
            palette={0: NO_PURCHASE_COLOR, 1: PURCHASE_COLOR},
        )
        axis.set_title(column)
        axis.set_xlabel("")
        legend = axis.get_legend()
        if legend is not None:
            legend.set_title("")
            for text, label in zip(legend.texts, ["No purchase", "Purchase"], strict=True):
                text.set_text(label)

    figure.suptitle("Numerical features: buyers vs non-buyers (density, top 1% cut)", fontsize=14)
    figure.tight_layout()
    return save_figure(figure, "numerical_distributions.png")


def plot_categorical_purchase_rates(data):
    """
    For each categorical feature, show the purchase rate for every value.

    For example: what share of November sessions ended in a purchase,
    compared with February sessions. The dashed line is the overall rate.

    Input:
        data: the full DataFrame.
    Returns:
        The path of the saved plot.
    """
    overall_rate = data[TARGET_COLUMN].mean()
    month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    figure, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes = axes.flatten()

    for index, column in enumerate(CATEGORICAL_FEATURES):
        axis = axes[index]
        rates = data.groupby(column)[TARGET_COLUMN].mean()
        if column == "Month":
            rates = rates.reindex([month for month in month_order if month in rates.index])
        axis.bar([str(value) for value in rates.index], rates.values, color=PURCHASE_COLOR)
        axis.axhline(overall_rate, color="black", linestyle="--", linewidth=1)
        axis.set_title(column)
        axis.set_ylabel("Purchase rate")
        axis.tick_params(axis="x", rotation=45)

    # 7 categorical features, 8 boxes: hide the empty one.
    axes[-1].axis("off")
    figure.suptitle("Purchase rate by category (dashed line = overall rate)", fontsize=14)
    figure.tight_layout()
    return save_figure(figure, "categorical_purchase_rates.png")


def plot_correlation_heatmap(data):
    """
    Heatmap of the correlation between the numerical features and the target.

    Correlation goes from -1 to +1. Values near +1 or -1 mean two columns
    move together. It helps us spot features that carry the same
    information (for example BounceRates and ExitRates) and features that
    are related to buying (PageValues stands out).

    Input:
        data: the full DataFrame.
    Returns:
        The path of the saved plot.
    """
    columns = NUMERICAL_FEATURES + [TARGET_COLUMN]
    correlation = data[columns].corr()

    figure, axis = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        correlation,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        square=True,
        cbar_kws={"shrink": 0.8},
        ax=axis,
    )
    axis.set_title("Correlation between numerical features and Revenue")
    return save_figure(figure, "correlation_heatmap.png")


def run_eda(data):
    """
    Run all the EDA steps and save every plot to outputs/figures/.

    Input:
        data: the full DataFrame.
    Returns:
        A list with the paths of the saved plots.
    """
    print("\nDrawing EDA plots...")
    saved_paths = [
        plot_class_balance(data),
        plot_numerical_distributions(data),
        plot_categorical_purchase_rates(data),
        plot_correlation_heatmap(data),
    ]
    for path in saved_paths:
        print("  Saved", path)
    return saved_paths


In [ ]:
dataset = load_dataset()
quality_report = validate_dataset(dataset)
dataset.head()

## 2. Exploratory data analysis

Four questions: how imbalanced is the target, how do the numerical features differ between buyers and non-buyers, which categories buy more often, and which features move together?

In [ ]:
figure_paths = run_eda(dataset)
for path in figure_paths:
    show(path)

**What the EDA shows**

- Only about 15% of sessions buy. A model that always says "no purchase" is 85% accurate and useless, so we judge models with **precision, recall, F1 and ROC-AUC** instead of accuracy.
- **PageValues** separates the classes most clearly: most non-buyers have a page value of 0, most buyers do not. It is also the numerical feature most correlated with `Revenue`.
- **BounceRates and ExitRates** are strongly correlated with each other, and both are higher for non-buyers.
- **November** has the highest purchase rate, and new visitors buy more often than returning ones.
- The duration features are very skewed (many zeros, a long tail), which matters for distance-based KNN.

## 3. Train/test split and preprocessing

We split 80/20, **stratified** so both parts keep the 15% buyer rate, with `random_state=42` everywhere for reproducibility. The test set is put aside until the final evaluation.

Preprocessing lives inside a scikit-learn `Pipeline` with a `ColumnTransformer`:
- categorical columns (Month, VisitorType, codes such as Browser) are **one-hot encoded**;
- numerical columns are **standardised** for KNN and Naive Bayes (KNN compares distances, so a duration in thousands of seconds would otherwise drown out a rate between 0 and 0.2). Tree models do not need scaling, so we keep their numbers in real units, which keeps the tree's split rules readable.

Because the scaler and encoder are fitted inside the pipeline, they only ever see training data, even inside each cross-validation fold. That prevents **data leakage**.

<details><summary>The code below is <code>predator/preprocess.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
preprocess.py - Turning raw session data into numbers the models can use.

Models only understand numbers, and some models (KNN) also need those
numbers to be on a similar scale. This file builds scikit-learn Pipelines
that do the preprocessing and the model in one object.

Why a Pipeline? It makes data leakage much harder. Data leakage means
information from the test set (or from the validation fold during
cross-validation) sneaks into training. For example, if we computed the
mean and standard deviation for scaling on the WHOLE dataset, the test set
would have influenced training. Inside a Pipeline, the scaler and encoder
are fitted only on the data passed to .fit(), which is always training data.
"""

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler



def build_preprocessor(numerical_features, categorical_features, scale_numbers=True):
    """
    Build a ColumnTransformer that prepares numerical and categorical columns.

    - Categorical columns are one-hot encoded: the column Month becomes
      Month_Feb, Month_Mar, ... with a 1 in the matching column and 0
      elsewhere. This avoids pretending that "Dec" > "Feb".
      handle_unknown="ignore" means a value never seen in training (for
      example "Jan") just gets all zeros instead of crashing.
    - Numerical columns are standardised (mean 0, standard deviation 1)
      when scale_numbers is True. KNN measures distances, so without
      scaling a column measured in thousands of seconds
      (ProductRelated_Duration) would drown out a rate between 0 and 0.2
      (BounceRates).

    Tree models (Decision Tree, XGBoost) split on one column at a time,
    so scaling does not change them. We skip it for trees so the split
    rules stay in real units, e.g. "PageValues <= 0.94".

    Inputs:
        numerical_features: list of numerical column names.
        categorical_features: list of categorical column names.
        scale_numbers: True to standardise numerical columns.
    Returns:
        An unfitted ColumnTransformer.
    """
    if scale_numbers:
        numerical_step = StandardScaler()
    else:
        numerical_step = "passthrough"

    categorical_step = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ("numbers", numerical_step, numerical_features),
            ("categories", categorical_step, categorical_features),
        ]
    )
    return preprocessor


def build_pipeline(model, numerical_features=None, categorical_features=None, scale_numbers=True):
    """
    Put the preprocessing and a model together in one Pipeline.

    Calling pipeline.fit(train_features, train_target) first fits the
    preprocessor on the training data and then trains the model on the
    transformed data. pipeline.predict(new_data) applies the exact same
    transformation before predicting.

    Inputs:
        model: an unfitted scikit-learn style classifier.
        numerical_features: numerical columns (defaults to all of them).
        categorical_features: categorical columns (defaults to all of them).
        scale_numbers: passed on to build_preprocessor().
    Returns:
        An unfitted Pipeline with two steps: "preprocess" and "model".
    """
    if numerical_features is None:
        numerical_features = NUMERICAL_FEATURES
    if categorical_features is None:
        categorical_features = CATEGORICAL_FEATURES

    preprocessor = build_preprocessor(numerical_features, categorical_features, scale_numbers)
    pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    return pipeline


def get_feature_names(fitted_pipeline):
    """
    Get readable names for the columns the model actually sees.

    After one-hot encoding there are many more columns than the 17
    original features, for example "VisitorType_New_Visitor".

    Input:
        fitted_pipeline: a Pipeline that has already been fitted.
    Returns:
        A list of column names, in the same order as the model's input.
    """
    raw_names = fitted_pipeline.named_steps["preprocess"].get_feature_names_out()
    # scikit-learn adds prefixes like "numbers__" and "categories__"; drop them.
    clean_names = [name.split("__", 1)[1] for name in raw_names]
    return clean_names


def undersample_training_data(train_features, train_target):
    """
    Randomly undersample the majority class (non-buyers) in the training data.

    We keep every buyer and pick an equal number of non-buyers at random,
    so the result is 50% buyers / 50% non-buyers. This helps KNN, because
    otherwise most of a session's nearest neighbours are non-buyers simply
    because there are so many of them.

    The downside: we throw away most of the non-buyer sessions, so the
    model learns from less data. This is only ever applied to TRAINING
    data. The test set keeps its real 15% buyer rate.

    Inputs:
        train_features: DataFrame of training features.
        train_target: Series of training labels (0/1).
    Returns:
        (undersampled_features, undersampled_target), shuffled.
    """
    buyer_rows = train_target[train_target == 1].index
    non_buyer_rows = train_target[train_target == 0].index

    sampled_non_buyer_rows = pd.Series(non_buyer_rows).sample(n=len(buyer_rows), random_state=RANDOM_STATE)

    kept_rows = list(buyer_rows) + list(sampled_non_buyer_rows)
    undersampled_features = train_features.loc[kept_rows]
    undersampled_target = train_target.loc[kept_rows]

    # Shuffle so buyers and non-buyers are mixed (order should not matter,
    # but it makes the data look like a normal random sample).
    shuffled_order = undersampled_features.sample(frac=1, random_state=RANDOM_STATE).index
    return undersampled_features.loc[shuffled_order], undersampled_target.loc[shuffled_order]


In [ ]:
train_features, test_features, train_target, test_target = split_data(dataset)

## 4. The models and their hyperparameter grids

- **Decision Tree**: asks yes/no questions, choosing at each step the split that lowers Gini impurity or entropy the most. `class_weight="balanced"` makes errors on the rare buyer class count more.
- **KNN**: classifies a session by a vote of its k most similar training sessions (Euclidean or Manhattan distance). It has no class weights, so we also train a version on **randomly undersampled** training data (all buyers plus an equal number of non-buyers).
- **Gaussian Naive Bayes**: applies Bayes' theorem assuming features are independent and bell-shaped within each class. Neither assumption holds here, which makes it a useful untuned baseline.
- **XGBoost (benchmark)**: gradient boosting; each new small tree is trained to fix the errors of the trees before it. `scale_pos_weight` (non-buyers / buyers) handles the imbalance.

<details><summary>The code below is <code>predator/models.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
models.py - The four models and the settings we try for each one.

Main models (from the AI Lab course):
    - Decision Tree
    - K-Nearest Neighbours (KNN), trained two ways: on all training data,
      and on randomly undersampled (balanced) training data
Baseline:
    - Gaussian Naive Bayes (no tuning, it shows the score to beat)
Benchmark beyond the course:
    - XGBoost, to see how far a stronger off-the-shelf model gets

Each model comes with a "parameter grid": the hyperparameter values that
GridSearchCV will try. Hyperparameters are settings we choose before
training (like the tree depth), unlike parameters the model learns itself.
The names start with "model__" because the model sits inside a Pipeline
step called "model".
"""

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


# Model names used everywhere (tables, plots, saved files, the app).
DECISION_TREE = "Decision Tree"
KNN = "KNN"
KNN_UNDERSAMPLED = "KNN (undersampled)"
NAIVE_BAYES = "Naive Bayes (baseline)"
XGBOOST = "XGBoost (benchmark)"

MODEL_NAMES = [DECISION_TREE, KNN, KNN_UNDERSAMPLED, NAIVE_BAYES, XGBOOST]

# Short file-friendly names for saving models with joblib.
MODEL_FILE_NAMES = {
    DECISION_TREE: "decision_tree.joblib",
    KNN: "knn.joblib",
    KNN_UNDERSAMPLED: "knn_undersampled.joblib",
    NAIVE_BAYES: "naive_bayes.joblib",
    XGBOOST: "xgboost_benchmark.joblib",
}

# What each model is there for, shown in the comparison table.
MODEL_ROLES = {
    DECISION_TREE: "Main (course)",
    KNN: "Main (course)",
    KNN_UNDERSAMPLED: "Main (course)",
    NAIVE_BAYES: "Baseline",
    XGBOOST: "Benchmark (beyond course)",
}

# One colour per model, so every plot uses the same colour for a model.
MODEL_COLORS = {
    DECISION_TREE: "#2A9D8F",
    KNN: "#E76F51",
    KNN_UNDERSAMPLED: "#F4A261",
    NAIVE_BAYES: "#8D99AE",
    XGBOOST: "#264653",
}


def make_decision_tree():
    """
    Create an unfitted Decision Tree classifier.

    How it works: the tree asks yes/no questions about the features
    ("Is PageValues <= 0.94?"). At every step it picks the question that
    best separates buyers from non-buyers. "Best" is measured with an
    impurity score:
        - Gini impurity: the chance that two sessions picked at random from
          the node have different labels. 0 means the node is pure.
        - Entropy: how "surprised" we are on average by a label in the
          node, from information theory. Also 0 when the node is pure.
    The split that lowers the impurity the most wins.

    class_weight="balanced" makes mistakes on the rare buyer class count
    more (about 5.5 times more here), so the tree does not just learn to
    say "no purchase" for everything.

    Returns:
        A DecisionTreeClassifier.
    """
    return DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)


def make_knn():
    """
    Create an unfitted K-Nearest Neighbours classifier.

    How it works: KNN does not really "train". It remembers all training
    sessions. To classify a new session it finds the k most similar
    training sessions (its nearest neighbours) and lets them vote.
    Similarity is measured with a distance:
        - Euclidean: straight-line distance, sqrt(sum of squared differences)
        - Manhattan: sum of absolute differences (like walking city blocks)
    Because it is all about distances, features must be scaled first.
    weights="distance" lets closer neighbours have a bigger vote.

    KNN has no class_weight option, which is why we also try it on
    undersampled training data.

    Returns:
        A KNeighborsClassifier.
    """
    return KNeighborsClassifier()


def make_naive_bayes():
    """
    Create an unfitted Gaussian Naive Bayes classifier.

    How it works: it uses Bayes' theorem to compute
    P(purchase | features) from P(features | purchase) and P(purchase).
    The "naive" assumption is that all features are independent of each
    other once you know the class, and "Gaussian" means each numerical
    feature is assumed to follow a bell curve within each class.
    Neither is true here (ProductRelated and ProductRelated_Duration are
    strongly linked, and most features are very skewed), so we use it as a
    simple, fast baseline that the other models should beat.

    Returns:
        A GaussianNB classifier.
    """
    return GaussianNB()


def make_xgboost(scale_pos_weight):
    """
    Create an unfitted XGBoost classifier (benchmark beyond the course).

    How it works: gradient boosting builds many small decision trees one
    after another. Each new tree is trained to fix the mistakes (the
    remaining errors) of all the trees before it, and the final prediction
    adds up all the trees. learning_rate controls how big a step each new
    tree takes.

    scale_pos_weight tells XGBoost to weigh buyer sessions more, set to
    (number of non-buyers / number of buyers) in the training data. It is
    the XGBoost version of class_weight="balanced".

    Input:
        scale_pos_weight: weight for the positive (buyer) class.
    Returns:
        An XGBClassifier.
    """
    return XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )


def get_parameter_grids():
    """
    The hyperparameter values GridSearchCV will try for each model.

    GridSearchCV tries every combination, so the grids are kept small
    enough to finish in a few minutes on a laptop.

    Returns:
        A dictionary: model name -> parameter grid (dictionary of lists).
        Naive Bayes has an empty grid because it is an untuned baseline.
    """
    decision_tree_grid = {
        "model__max_depth": [3, 5, 7, 10, 15, None],
        "model__min_samples_split": [2, 10, 20, 50],
        "model__min_samples_leaf": [1, 5, 10, 20],
        "model__criterion": ["gini", "entropy"],
    }

    # Odd values of k (1, 3, 5, ..., 31) avoid ties in a two-class vote.
    knn_grid = {
        "model__n_neighbors": list(range(1, 32, 2)),
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    }

    xgboost_grid = {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 4, 5, 6],
        "model__learning_rate": [0.03, 0.1, 0.3],
    }

    grids = {
        DECISION_TREE: decision_tree_grid,
        KNN: knn_grid,
        KNN_UNDERSAMPLED: knn_grid,
        NAIVE_BAYES: {},
        XGBOOST: xgboost_grid,
    }
    return grids


def uses_scaling(model_name):
    """
    Say whether a model needs its numerical features scaled.

    Input:
        model_name: one of MODEL_NAMES.
    Returns:
        True for distance/probability models (KNN, Naive Bayes),
        False for tree models (Decision Tree, XGBoost).
    """
    tree_models = [DECISION_TREE, XGBOOST]
    return model_name not in tree_models


def make_model(model_name, scale_pos_weight):
    """
    Create a fresh, unfitted model from its name.

    Inputs:
        model_name: one of MODEL_NAMES.
        scale_pos_weight: only used by XGBoost.
    Returns:
        The unfitted model object.
    """
    if model_name == DECISION_TREE:
        return make_decision_tree()
    if model_name in (KNN, KNN_UNDERSAMPLED):
        return make_knn()
    if model_name == NAIVE_BAYES:
        return make_naive_bayes()
    if model_name == XGBOOST:
        return make_xgboost(scale_pos_weight)
    raise ValueError("Unknown model name: " + model_name)


In [ ]:
for model_name, grid in get_parameter_grids().items():
    print(model_name, "->", grid if grid else "no tuning (baseline)")

## 5. Tuning with 5-fold stratified cross-validation

`GridSearchCV` tries every combination in each grid, training on 4 folds and scoring F1 on the 5th, and keeps the best average. Afterwards we compute **out-of-fold** probabilities for every training session (each one predicted by a model that did not see it). These give a fair cross-validated F1 for every model on the real class balance, including the undersampled KNN, and they are what we later use to choose decision thresholds without touching the test set.

<details><summary>The code below is <code>predator/train.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
train.py - Tuning the models with cross-validation and saving them.

Cross-validation in one paragraph: we split the TRAINING data into 5 equal
parts (folds). We train on 4 folds and score on the 5th, and repeat so
every fold is used for scoring once. The average of the 5 scores tells us
how well a setting generalises, without ever touching the test set.
"Stratified" folds keep the 15% buyer rate in every fold.

GridSearchCV runs this for every combination in a parameter grid and keeps
the combination with the best average F1 score. We use F1 (not accuracy)
because accuracy is misleading on imbalanced data: always saying
"no purchase" scores about 85% accuracy but finds zero buyers. F1 is the
balance (harmonic mean) of precision and recall on the buyer class.
"""

import json
import os
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold


MODELS_FOLDER = "models"
NUMBER_OF_FOLDS = 5


def make_folds():
    """
    Create the 5-fold stratified cross-validation splitter used everywhere.

    shuffle=True with a fixed random_state mixes the rows before splitting,
    and gives the same folds on every run.

    Returns:
        A StratifiedKFold object.
    """
    return StratifiedKFold(n_splits=NUMBER_OF_FOLDS, shuffle=True, random_state=RANDOM_STATE)


def calculate_scale_pos_weight(train_target):
    """
    Work out how much more weight each buyer should get in XGBoost.

    Input:
        train_target: Series of training labels (0/1).
    Returns:
        number of non-buyers divided by number of buyers (about 5.5 here).
    """
    number_of_buyers = int((train_target == 1).sum())
    number_of_non_buyers = int((train_target == 0).sum())
    return number_of_non_buyers / number_of_buyers


def tune_model(model_name, train_features, train_target, numerical_features, categorical_features):
    """
    Find the best hyperparameters for one model with GridSearchCV.

    For "KNN (undersampled)" the training data is undersampled first, so
    this KNN learns from a 50/50 balanced set. For Naive Bayes the grid is
    empty, so the model is simply fitted once (it is an untuned baseline).

    Inputs:
        model_name: one of MODEL_NAMES.
        train_features, train_target: the training data.
        numerical_features, categorical_features: which columns to use.
    Returns:
        A dictionary with:
            "pipeline": the best pipeline, refitted on the training data
            "best_params": the winning hyperparameters
            "grid_f1": the best mean F1 inside GridSearchCV (None for Naive Bayes)
    """
    scale_pos_weight = calculate_scale_pos_weight(train_target)
    model = make_model(model_name, scale_pos_weight)
    pipeline = build_pipeline(
        model, numerical_features, categorical_features, scale_numbers=uses_scaling(model_name)
    )
    parameter_grid = get_parameter_grids()[model_name]

    fit_features = train_features
    fit_target = train_target
    if model_name == KNN_UNDERSAMPLED:
        fit_features, fit_target = undersample_training_data(train_features, train_target)
        print("  Undersampled training set:", len(fit_features), "sessions (50% buyers)")

    # Naive Bayes: nothing to tune, just fit it.
    if len(parameter_grid) == 0:
        pipeline.fit(fit_features, fit_target)
        return {"pipeline": pipeline, "best_params": {}, "grid_f1": None}

    number_of_combinations = 1
    for values in parameter_grid.values():
        number_of_combinations = number_of_combinations * len(values)
    print("  Trying", number_of_combinations, "combinations x", NUMBER_OF_FOLDS, "folds")

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring="f1",
        cv=make_folds(),
        n_jobs=-1,
    )
    grid_search.fit(fit_features, fit_target)

    # Remove the "model__" prefix so the parameters are easier to read.
    best_params = {}
    for key, value in grid_search.best_params_.items():
        best_params[key.replace("model__", "")] = value

    return {
        "pipeline": grid_search.best_estimator_,
        "best_params": best_params,
        "grid_f1": float(grid_search.best_score_),
    }


def out_of_fold_probabilities(pipeline, train_features, train_target, undersample=False):
    """
    Get a purchase probability for every training session from a model
    that did NOT see that session during training.

    This is cross-validation written out by hand: for each of the 5 folds,
    train a copy of the pipeline on the other 4 folds and predict the
    held-out fold. We use these "out-of-fold" probabilities to:
      1. compare all models fairly on the real 15% buyer rate (the
         undersampled KNN's GridSearchCV score was measured on balanced
         folds, so it is not comparable), and
      2. choose decision thresholds without looking at the test set.

    When undersample is True, only the 4 training folds are undersampled.
    The held-out fold keeps its real class balance.

    Inputs:
        pipeline: a pipeline with the chosen hyperparameters.
        train_features, train_target: the training data.
        undersample: True for the undersampled KNN.
    Returns:
        A numpy array with one probability per training session.
    """
    probabilities = np.zeros(len(train_features))

    for fold_train_rows, fold_validation_rows in make_folds().split(train_features, train_target):
        fold_train_features = train_features.iloc[fold_train_rows]
        fold_train_target = train_target.iloc[fold_train_rows]
        fold_validation_features = train_features.iloc[fold_validation_rows]

        if undersample:
            fold_train_features, fold_train_target = undersample_training_data(
                fold_train_features, fold_train_target
            )

        # clone() gives an unfitted copy with the same settings.
        fold_model = clone(pipeline)
        fold_model.fit(fold_train_features, fold_train_target)
        probabilities[fold_validation_rows] = fold_model.predict_proba(fold_validation_features)[:, 1]

    return probabilities


def train_all_models(train_features, train_target, numerical_features=None, categorical_features=None):
    """
    Tune, train and cross-validate every model.

    Inputs:
        train_features, train_target: the training data.
        numerical_features, categorical_features: which columns to use
            (defaults to all; the ablation passes a list without PageValues).
    Returns:
        A dictionary: model name -> result dictionary with keys
        "pipeline", "best_params", "grid_f1", "oof_probabilities",
        "cv_f1" (fair 5-fold F1 at threshold 0.5) and "training_seconds".
    """
    if numerical_features is None:
        numerical_features = NUMERICAL_FEATURES
    if categorical_features is None:
        categorical_features = CATEGORICAL_FEATURES

    results = {}
    for model_name in MODEL_NAMES:
        print("\nTraining " + model_name + "...")
        start_time = time.time()

        result = tune_model(
            model_name, train_features, train_target, numerical_features, categorical_features
        )

        undersample = model_name == KNN_UNDERSAMPLED
        oof_probabilities = out_of_fold_probabilities(
            result["pipeline"], train_features, train_target, undersample
        )
        oof_predictions = (oof_probabilities >= 0.5).astype(int)
        result["oof_probabilities"] = oof_probabilities
        result["cv_f1"] = float(f1_score(train_target, oof_predictions))
        result["training_seconds"] = round(time.time() - start_time, 1)

        if len(result["best_params"]) > 0:
            print("  Best settings:", result["best_params"])
        print("  Cross-validated F1 (real class balance): {:.3f}".format(result["cv_f1"]))
        print("  Took", result["training_seconds"], "seconds")
        results[model_name] = result

    return results


def save_models(results, folder=MODELS_FOLDER, file_prefix=""):
    """
    Save every trained pipeline to disk with joblib.

    The whole pipeline is saved (preprocessing + model), so loading it
    later gives an object that accepts raw session data directly.

    Inputs:
        results: the dictionary returned by train_all_models().
        folder: where to save the files.
        file_prefix: added to each file name (e.g. "no_pagevalues_").
    Returns:
        A dictionary: model name -> saved file path.
    """
    os.makedirs(folder, exist_ok=True)
    saved_paths = {}
    for model_name, result in results.items():
        path = os.path.join(folder, file_prefix + MODEL_FILE_NAMES[model_name])
        joblib.dump(result["pipeline"], path)
        saved_paths[model_name] = path
        print("  Saved", path)
    return saved_paths


OUT_OF_FOLD_FILE = "oof_probabilities.csv"


def save_out_of_fold_probabilities(training_results, train_target, folder=MODELS_FOLDER):
    """
    Save every model's out-of-fold training probabilities with the real labels.

    The app uses them to find the best-profit threshold again when the
    user changes the business assumptions, without touching the test set.

    Inputs:
        training_results: from train_all_models().
        train_target: the training labels.
        folder: where to save the CSV.
    Returns:
        The path of the saved file.
    """
    table = pd.DataFrame({"Revenue": np.asarray(train_target)})
    for model_name, result in training_results.items():
        table[model_name] = np.round(result["oof_probabilities"], 4)
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, OUT_OF_FOLD_FILE)
    table.to_csv(path, index=False)
    return path


def load_out_of_fold_probabilities(folder=MODELS_FOLDER):
    """
    Load the file saved by save_out_of_fold_probabilities().

    Returns:
        A DataFrame with a "Revenue" column and one probability column per model.
    """
    return pd.read_csv(os.path.join(folder, OUT_OF_FOLD_FILE))


def save_model_info(model_info, folder=MODELS_FOLDER):
    """
    Save extra information about the trained models as JSON.

    This holds things the prediction script and the app need: the best
    model's name, the chosen thresholds, default input values and the
    permutation importance results.

    Inputs:
        model_info: a dictionary that can be written as JSON.
        folder: the models folder.
    Returns:
        The path of the saved JSON file.
    """
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, "model_info.json")
    with open(path, "w") as file:
        json.dump(model_info, file, indent=2)
    return path


def load_model_info(folder=MODELS_FOLDER):
    """
    Load the JSON file written by save_model_info().

    Input:
        folder: the models folder.
    Returns:
        The model_info dictionary.
    Raises:
        FileNotFoundError if the file is missing (run main.py first).
    """
    path = os.path.join(folder, "model_info.json")
    if not os.path.isfile(path):
        raise FileNotFoundError(path + " not found. Run 'python main.py' first.")
    with open(path) as file:
        return json.load(file)


In [ ]:
training_results = train_all_models(train_features, train_target)
models = {name: result["pipeline"] for name, result in training_results.items()}

## 6. Evaluation on the test set

Every model is scored once on the untouched test set at the default 0.5 threshold. The best model is chosen by **cross-validated F1** (choosing by test score would quietly tune on the test set).

<details><summary>The code below is <code>predator/evaluate.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
evaluate.py - Scoring the models on the test set and drawing result plots.

The metrics, in plain words (a "positive" is a session that buys):
    - Accuracy: share of all sessions predicted correctly. Misleading here,
      because always saying "no purchase" already gets about 85%.
    - Precision: of the sessions we flagged as buyers, how many really bought.
    - Recall: of the real buyers, how many we found.
    - F1: one number that balances precision and recall (their harmonic mean).
    - ROC-AUC: the chance that a random buyer gets a higher probability than
      a random non-buyer. 0.5 is guessing, 1.0 is perfect. It does not
      depend on the 0.5 decision threshold.
    - Confusion matrix: a 2x2 table of true/false positives and negatives.
"""

import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


OUTPUTS_FOLDER = "outputs"


def get_purchase_probabilities(pipeline, features):
    """
    Get the predicted probability of a purchase for each session.

    predict_proba() returns two columns: P(no purchase) and P(purchase).
    We keep the second one.

    Inputs:
        pipeline: a fitted pipeline.
        features: DataFrame of sessions.
    Returns:
        A numpy array of probabilities between 0 and 1.
    """
    return pipeline.predict_proba(features)[:, 1]


def evaluate_model(model_name, pipeline, test_features, test_target, threshold=0.5):
    """
    Compute all the test-set metrics for one model.

    A session is predicted as a buyer when its probability is at least the
    threshold (0.5 by default; see threshold.py for choosing a better one).

    Inputs:
        model_name: the model's name (used in the report).
        pipeline: a fitted pipeline.
        test_features, test_target: the test data.
        threshold: the probability cutoff for predicting "purchase".
    Returns:
        A dictionary with the metrics, the confusion matrix and the
        classification report text.
    """
    probabilities = get_purchase_probabilities(pipeline, test_features)
    predictions = (probabilities >= threshold).astype(int)

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(test_target, predictions),
        "precision": precision_score(test_target, predictions, zero_division=0),
        "recall": recall_score(test_target, predictions),
        "f1": f1_score(test_target, predictions),
        "roc_auc": roc_auc_score(test_target, probabilities),
        "confusion_matrix": confusion_matrix(test_target, predictions),
        "report": classification_report(
            test_target, predictions, target_names=["No purchase", "Purchase"], digits=3
        ),
    }
    return metrics


def build_comparison_table(models, training_results, test_features, test_target, file_prefix=""):
    """
    Evaluate every model and put the scores in one table.

    It also prints each model's confusion matrix and classification report,
    and saves the reports to outputs/<file_prefix>classification_reports.txt.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        training_results: the dictionary from train.train_all_models()
            (used for the cross-validated F1 column).
        test_features, test_target: the test data.
        file_prefix: added to both file names, so the ablation run
            (file_prefix="ablation_") does not overwrite the main results.
    Returns:
        A pandas DataFrame with one row per model, also saved to
        outputs/<file_prefix>model_comparison.csv.
    """
    print("\nEvaluating models on the test set (threshold 0.5)...")
    rows = []
    report_text = ""

    for model_name, pipeline in models.items():
        metrics = evaluate_model(model_name, pipeline, test_features, test_target)
        rows.append(
            {
                "Model": model_name,
                "Role": MODEL_ROLES[model_name],
                "CV F1": training_results[model_name]["cv_f1"],
                "Accuracy": metrics["accuracy"],
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "F1": metrics["f1"],
                "ROC-AUC": metrics["roc_auc"],
            }
        )

        matrix = metrics["confusion_matrix"]
        section = (
            "=" * 60
            + "\n"
            + model_name
            + "\n"
            + "=" * 60
            + "\n"
            + "Confusion matrix (rows = actual, columns = predicted):\n"
            + "                 No purchase  Purchase\n"
            + f"  No purchase    {matrix[0][0]:>11}  {matrix[0][1]:>8}\n"
            + f"  Purchase       {matrix[1][0]:>11}  {matrix[1][1]:>8}\n\n"
            + metrics["report"]
            + "\n"
        )
        print(section)
        report_text = report_text + section

    table = pd.DataFrame(rows).round(3)

    os.makedirs(OUTPUTS_FOLDER, exist_ok=True)
    table.to_csv(os.path.join(OUTPUTS_FOLDER, file_prefix + "model_comparison.csv"), index=False)
    with open(os.path.join(OUTPUTS_FOLDER, file_prefix + "classification_reports.txt"), "w") as file:
        file.write(report_text)

    print("Model comparison (test set):")
    print(table.to_string(index=False))
    return table


def choose_best_model(comparison_table):
    """
    Pick the best model by cross-validated F1 on the training data.

    We do not pick by test-set score: the test set is only for reporting.
    Choosing the winner by test score would quietly tune on the test set.

    Input:
        comparison_table: the DataFrame from build_comparison_table().
    Returns:
        The name of the best model.
    """
    best_row = comparison_table.sort_values("CV F1", ascending=False).iloc[0]
    print(
        "\nBest model by cross-validated F1:", best_row["Model"], "(CV F1 = {:.3f})".format(best_row["CV F1"])
    )
    return best_row["Model"]


def plot_roc_curves(models, test_features, test_target, filename="roc_curves.png", title=None):
    """
    Draw the ROC curve of every model on one plot.

    A ROC curve shows the true positive rate (recall) against the false
    positive rate for every possible threshold. The closer a curve hugs
    the top-left corner, the better the model ranks buyers above
    non-buyers. The dashed diagonal is random guessing.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        test_features, test_target: the test data.
        filename: name of the image file.
        title: optional plot title.
    Returns:
        The path of the saved plot.
    """
    figure, axis = plt.subplots(figsize=(7, 6))
    for model_name, pipeline in models.items():
        probabilities = get_purchase_probabilities(pipeline, test_features)
        # roc_curve tries every threshold and returns the false positive
        # rate and true positive rate at each one.
        false_positive_rate, true_positive_rate, thresholds = roc_curve(test_target, probabilities)
        area = roc_auc_score(test_target, probabilities)
        axis.plot(
            false_positive_rate,
            true_positive_rate,
            color=MODEL_COLORS[model_name],
            linewidth=2,
            label=f"{model_name} (AUC = {area:.3f})",
        )
    axis.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1, label="Random guess")
    axis.set_xlabel("False positive rate (non-buyers wrongly flagged)")
    axis.set_ylabel("True positive rate (buyers found)")
    axis.set_title(title or "ROC curves on the test set")
    axis.legend(loc="lower right")
    return save_figure(figure, filename)


def plot_confusion_matrices(models, test_features, test_target, threshold=0.5):
    """
    Draw a confusion matrix for every model, side by side.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        test_features, test_target: the test data.
        threshold: the probability cutoff for predicting "purchase".
    Returns:
        The path of the saved plot.
    """
    figure, axes = plt.subplots(1, len(models), figsize=(4.2 * len(models), 4))
    for axis, (model_name, pipeline) in zip(axes, models.items(), strict=True):
        probabilities = get_purchase_probabilities(pipeline, test_features)
        predictions = (probabilities >= threshold).astype(int)
        ConfusionMatrixDisplay.from_predictions(
            test_target,
            predictions,
            display_labels=["No buy", "Buy"],
            cmap="Blues",
            colorbar=False,
            ax=axis,
        )
        axis.set_title(model_name)
    figure.suptitle(f"Confusion matrices on the test set (threshold {threshold})")
    figure.tight_layout()
    return save_figure(figure, "confusion_matrices.png")


def plot_metric_comparison(comparison_table):
    """
    Grouped bar chart of precision, recall, F1 and ROC-AUC for every model.

    Input:
        comparison_table: the DataFrame from build_comparison_table().
    Returns:
        The path of the saved plot.
    """
    metric_names = ["Precision", "Recall", "F1", "ROC-AUC"]
    number_of_models = len(comparison_table)
    bar_width = 0.8 / number_of_models

    figure, axis = plt.subplots(figsize=(10, 5))
    for index, row in comparison_table.reset_index(drop=True).iterrows():
        positions = [metric_index + index * bar_width for metric_index in range(len(metric_names))]
        values = [row[metric] for metric in metric_names]
        axis.bar(positions, values, width=bar_width, label=row["Model"], color=MODEL_COLORS[row["Model"]])

    axis.set_xticks([metric_index + 0.4 - bar_width / 2 for metric_index in range(len(metric_names))])
    axis.set_xticklabels(metric_names)
    axis.set_ylim(0, 1)
    axis.set_ylabel("Score on the test set")
    axis.set_title("Model comparison (threshold 0.5)")
    axis.legend(loc="upper left", fontsize=9)
    return save_figure(figure, "metric_comparison.png")


SLICE_COLUMNS = ["VisitorType", "Month", "Region"]


def evaluate_slices(models, model_names, test_features, test_target, threshold=0.5):
    """
    Score the models separately on groups of sessions ("slices"), e.g.
    only new visitors, or only November.

    One overall F1 can hide a group the model serves badly. Slices with
    few buyers give noisy scores, so the buyer count is kept next to
    every score.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        model_names: the models to analyse.
        test_features, test_target: the test data.
        threshold: the probability cutoff.
    Returns:
        A DataFrame with one row per model, slice column and value, also
        saved to outputs/slice_analysis.csv.
    """
    print("\nScoring the models on slices of the test set...")
    true_target = test_target.to_numpy()
    rows = []
    for model_name in model_names:
        predictions = (get_purchase_probabilities(models[model_name], test_features) >= threshold).astype(int)
        for column in SLICE_COLUMNS:
            for value in sorted(test_features[column].unique()):
                in_slice = (test_features[column] == value).to_numpy()
                slice_target = true_target[in_slice]
                slice_predictions = predictions[in_slice]
                rows.append(
                    {
                        "Model": model_name,
                        "Slice": column,
                        "Value": str(value),
                        "Sessions": int(in_slice.sum()),
                        "Buyers": int(slice_target.sum()),
                        "Purchase rate": slice_target.mean(),
                        "Precision": precision_score(slice_target, slice_predictions, zero_division=0),
                        "Recall": recall_score(slice_target, slice_predictions, zero_division=0),
                        "F1": f1_score(slice_target, slice_predictions, zero_division=0),
                    }
                )

    table = pd.DataFrame(rows).round(3)
    table.to_csv(os.path.join(OUTPUTS_FOLDER, "slice_analysis.csv"), index=False)
    plot_slice_scores(table, model_names)
    return table


def plot_slice_scores(slice_table, model_names):
    """
    Bar charts of F1 per visitor type and per month, one bar per model.

    Inputs:
        slice_table: the DataFrame from evaluate_slices().
        model_names: the models in the table.
    Returns:
        The path of the saved plot.
    """
    month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    figure, axes = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1, 3]})
    for axis, column in zip(axes, ["VisitorType", "Month"], strict=True):
        rows = slice_table[slice_table["Slice"] == column]
        values = list(rows["Value"].unique())
        if column == "Month":
            values = [month for month in month_order if month in values]
        bar_width = 0.8 / len(model_names)
        for index, model_name in enumerate(model_names):
            model_rows = rows[rows["Model"] == model_name].set_index("Value").loc[values]
            positions = [position + index * bar_width for position in range(len(values))]
            axis.bar(
                positions, model_rows["F1"], width=bar_width, color=MODEL_COLORS[model_name], label=model_name
            )
        buyer_counts = rows[rows["Model"] == model_names[0]].set_index("Value").loc[values, "Buyers"]
        labels = [f"{value}\n({count} buyers)" for value, count in zip(values, buyer_counts, strict=True)]
        axis.set_xticks(
            [position + bar_width * (len(model_names) - 1) / 2 for position in range(len(values))]
        )
        axis.set_xticklabels(labels, fontsize=8)
        axis.set_ylim(0, 1)
        axis.set_title("F1 by " + column, fontsize=11)
    axes[0].set_ylabel("F1 (test set, threshold 0.5)")
    axes[1].legend(fontsize=9)
    figure.tight_layout()
    return save_figure(figure, "slice_scores.png")


In [ ]:
comparison_table = build_comparison_table(models, training_results, test_features, test_target)
best_model_name = choose_best_model(comparison_table)
show(plot_roc_curves(models, test_features, test_target), width=600)
show(plot_confusion_matrices(models, test_features, test_target))
show(plot_metric_comparison(comparison_table), width=750)
comparison_table

**Reading the results**

- **XGBoost (benchmark)** and the **Decision Tree** are clearly ahead, with test ROC-AUC around 0.92 to 0.93. The tree gets most of the benchmark's performance.
- **KNN** struggles: with dozens of one-hot columns and skewed durations, "nearest" sessions are not very informative. Undersampling raises its recall a lot at the cost of precision, and improves its F1 and ROC-AUC.
- **Naive Bayes** flags almost every session as a buyer: its independence assumption breaks on correlated, skewed features, and its probabilities end up pushed to 0 or 1.

## 7. How sure are we? Confidence intervals, McNemar's test and slices

The test set has only 382 buyers, so every score carries some luck. Three checks:

- **Bootstrap 95% confidence intervals:** resample the test set with replacement 1,000 times, re-score each model, keep the middle 95%.
- **McNemar's test:** for two models on the same sessions, count the sessions only one of them gets right. If the models were equally good, the two counts would be about equal.
- **Slices:** scores per visitor type and month. One overall F1 can hide a group the model serves badly.

<details><summary>The code below is <code>predator/significance.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
significance.py - Are the differences between models real or just luck?

The test set has 2,466 sessions and only 382 buyers. A different random
test set would give slightly different scores, so a gap of 0.02 F1 may
be noise. Two standard tools tell us:

1. Bootstrap confidence intervals. We draw 1,000 new "test sets" of the
   same size by sampling the real test sessions WITH replacement, score
   the model on each, and take the middle 95% of those scores. If two
   models' intervals overlap a lot, the data cannot separate them well.

2. McNemar's test for two models on the same sessions. Only the
   sessions where the models disagree matter:
       b = sessions model A gets right and model B gets wrong
       c = sessions model B gets right and model A gets wrong
   If the models were equally good, b and c would be about equal. The
   test statistic (|b - c| - 1)^2 / (b + c) follows a chi-squared
   distribution with 1 degree of freedom; the p-value is the chance of a
   difference at least this big if the models were really equal.
"""

import math
import os

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score


BOOTSTRAP_ROUNDS = 1000
CONFIDENCE_LEVEL = 0.95


def bootstrap_confidence_intervals(test_target, probabilities, threshold=0.5):
    """
    Bootstrap 95% confidence intervals for F1 and ROC-AUC.

    Inputs:
        test_target: the real labels (0/1).
        probabilities: the model's purchase probabilities.
        threshold: the cutoff used for F1.
    Returns:
        A dictionary with "f1_low", "f1_high", "roc_auc_low", "roc_auc_high".
    """
    true_target = np.asarray(test_target)
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= threshold).astype(int)
    random_generator = np.random.default_rng(RANDOM_STATE)

    f1_scores = []
    roc_auc_scores = []
    for _ in range(BOOTSTRAP_ROUNDS):
        rows = random_generator.integers(0, len(true_target), size=len(true_target))
        f1_scores.append(f1_score(true_target[rows], predictions[rows], zero_division=0))
        roc_auc_scores.append(roc_auc_score(true_target[rows], probabilities[rows]))

    lower = (1 - CONFIDENCE_LEVEL) / 2 * 100
    upper = (1 + CONFIDENCE_LEVEL) / 2 * 100
    return {
        "f1_low": float(np.percentile(f1_scores, lower)),
        "f1_high": float(np.percentile(f1_scores, upper)),
        "roc_auc_low": float(np.percentile(roc_auc_scores, lower)),
        "roc_auc_high": float(np.percentile(roc_auc_scores, upper)),
    }


def mcnemar_test(test_target, predictions_a, predictions_b):
    """
    McNemar's test (with continuity correction) for two models' predictions.

    For a chi-squared distribution with 1 degree of freedom, the chance of
    a value at least x is erfc(sqrt(x / 2)), so the standard library is enough.

    Inputs:
        test_target: the real labels.
        predictions_a, predictions_b: the 0/1 predictions of the two models.
    Returns:
        A dictionary with "a_right_b_wrong", "b_right_a_wrong", "statistic"
        and "p_value".
    """
    true_target = np.asarray(test_target)
    a_right = np.asarray(predictions_a) == true_target
    b_right = np.asarray(predictions_b) == true_target
    a_right_b_wrong = int((a_right & ~b_right).sum())
    b_right_a_wrong = int((b_right & ~a_right).sum())

    disagreements = a_right_b_wrong + b_right_a_wrong
    if disagreements == 0:
        return {"a_right_b_wrong": 0, "b_right_a_wrong": 0, "statistic": 0.0, "p_value": 1.0}

    statistic = (abs(a_right_b_wrong - b_right_a_wrong) - 1) ** 2 / disagreements
    p_value = math.erfc(math.sqrt(statistic / 2))
    return {
        "a_right_b_wrong": a_right_b_wrong,
        "b_right_a_wrong": b_right_a_wrong,
        "statistic": float(statistic),
        "p_value": float(p_value),
    }


def run_significance_analysis(models, comparison_table, test_features, test_target, pairs_to_compare):
    """
    Add bootstrap intervals to the comparison table and run McNemar's test
    for each pair of models.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        comparison_table: the DataFrame from evaluate.build_comparison_table().
        test_features, test_target: the test data.
        pairs_to_compare: list of (model_a, model_b) name pairs.
    Returns:
        (intervals_table, mcnemar_table), saved to outputs/confidence_intervals.csv
        and outputs/mcnemar_tests.csv.
    """
    print(f"\nBootstrap confidence intervals ({BOOTSTRAP_ROUNDS} resamples of the test set)...")
    probabilities = {}
    rows = []
    for model_name, pipeline in models.items():
        probabilities[model_name] = get_purchase_probabilities(pipeline, test_features)
        intervals = bootstrap_confidence_intervals(test_target, probabilities[model_name])
        scores = comparison_table[comparison_table["Model"] == model_name].iloc[0]
        rows.append(
            {
                "Model": model_name,
                "F1": scores["F1"],
                "F1 95% low": intervals["f1_low"],
                "F1 95% high": intervals["f1_high"],
                "ROC-AUC": scores["ROC-AUC"],
                "ROC-AUC 95% low": intervals["roc_auc_low"],
                "ROC-AUC 95% high": intervals["roc_auc_high"],
            }
        )
    intervals_table = pd.DataFrame(rows).round(3)
    intervals_table.to_csv(os.path.join(OUTPUTS_FOLDER, "confidence_intervals.csv"), index=False)
    print(intervals_table.to_string(index=False))

    print("\nMcNemar's test (threshold 0.5): do two models make different mistakes?")
    rows = []
    for model_a, model_b in pairs_to_compare:
        result = mcnemar_test(
            test_target,
            (probabilities[model_a] >= 0.5).astype(int),
            (probabilities[model_b] >= 0.5).astype(int),
        )
        rows.append(
            {
                "Model A": model_a,
                "Model B": model_b,
                "A right, B wrong": result["a_right_b_wrong"],
                "B right, A wrong": result["b_right_a_wrong"],
                "Chi-squared": round(result["statistic"], 3),
                "p-value": round(result["p_value"], 4),
                "Significant at 5%": result["p_value"] < 0.05,
            }
        )
    mcnemar_table = pd.DataFrame(rows)
    mcnemar_table.to_csv(os.path.join(OUTPUTS_FOLDER, "mcnemar_tests.csv"), index=False)
    print(mcnemar_table.to_string(index=False))
    return intervals_table, mcnemar_table


In [ ]:
intervals_table, mcnemar_table = run_significance_analysis(
    models, comparison_table, test_features, test_target,
    pairs_to_compare=[(DECISION_TREE, XGBOOST), (DECISION_TREE, KNN_UNDERSAMPLED)])
slice_table = evaluate_slices(models, [DECISION_TREE, XGBOOST], test_features, test_target)
show("outputs/figures/slice_scores.png")
display(intervals_table, mcnemar_table)

**Reading the checks**

- The Decision Tree's F1 interval (0.59 to 0.66) overlaps XGBoost's (0.61 to 0.68), but McNemar's test still finds XGBoost **significantly better** (87 sessions only XGBoost gets right against 52, p = 0.004). Overlapping intervals are not the same as "no difference": both models are scored on the same sessions, and McNemar uses that pairing.
- The Decision Tree beats the undersampled KNN by a wide margin (p < 0.001).
- **Returning visitors are the hard group:** the tree's F1 is 0.59 for them against 0.77 for new visitors, because many returning visitors browse valuable pages without buying (precision 0.47). Months with only a handful of buyers (February has 1) give noisy scores.

## 8. Overfitting and underfitting: learning and validation curves

A **learning curve** trains the tuned model on growing parts of the training data. A **validation curve** changes one setting (tree depth, number of neighbours) and keeps the rest. Both use the same 5 folds and F1 as the grid search.

<details><summary>The code below is <code>predator/curves.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
curves.py - Overfitting and underfitting, shown with our own data.

Two diagnostic plots from the course:

1. Learning curve: train the tuned model on 10%, 28%, ... 100% of the
   training data and plot the training score and the cross-validated
   score. A big gap between them means overfitting (high variance): more
   data would help. Two low curves that meet mean underfitting (high
   bias): a more flexible model would help.

2. Validation curve: change ONE setting and keep the others at their
   tuned values. For the Decision Tree we change max_depth: a shallow
   tree underfits, a very deep tree memorises the training data. For
   KNN we change n_neighbors (k): k = 1 memorises every training session,
   a very large k averages everything away. The best value sits where
   the cross-validated score peaks.

All scores are F1 from 5-fold stratified cross-validation on the
training data, the same folds as GridSearchCV.
"""

import matplotlib

matplotlib.use("Agg")
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import learning_curve, validation_curve


TRAINING_SIZES = np.linspace(0.1, 1.0, 6)
TREE_DEPTHS_TO_TRY = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 15, 20]
NEIGHBOURS_TO_TRY = [1, 3, 5, 9, 15, 21, 31, 51, 75, 101]


def draw_score_band(axis, x_values, scores, color, line_style, label):
    """
    Plot the mean score across the 5 folds as a line, with a shaded band
    of plus/minus one standard deviation.

    Inputs:
        axis: the matplotlib axis.
        x_values: the x positions.
        scores: array of shape (number of x values, 5 folds).
        color, line_style, label: how to draw the line.
    """
    means = scores.mean(axis=1)
    deviations = scores.std(axis=1)
    axis.plot(x_values, means, color=color, linestyle=line_style, marker="o", label=label)
    axis.fill_between(x_values, means - deviations, means + deviations, color=color, alpha=0.15)


def plot_learning_curves(models, train_features, train_target):
    """
    Draw the learning curves of the Decision Tree and KNN.

    Inputs:
        models: dictionary model name -> fitted pipeline (the tuned settings are reused).
        train_features, train_target: the training data.
    Returns:
        A DataFrame with the mean scores, also saved to outputs/learning_curves.csv.
    """
    print("\nLearning curves (F1, 5-fold CV)...")
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    rows = []
    for axis, model_name in zip(axes, [DECISION_TREE, KNN], strict=True):
        sizes, train_scores, validation_scores = learning_curve(
            models[model_name],
            train_features,
            train_target,
            train_sizes=TRAINING_SIZES,
            cv=make_folds(),
            scoring="f1",
            n_jobs=-1,
        )
        color = MODEL_COLORS[model_name]
        draw_score_band(axis, sizes, train_scores, color, "--", "Training F1")
        draw_score_band(axis, sizes, validation_scores, color, "-", "Cross-validated F1")
        axis.set_title(model_name, fontsize=11)
        axis.set_xlabel("Number of training sessions")
        axis.legend(fontsize=9, loc="lower right")
        for size, train_score, validation_score in zip(
            sizes, train_scores.mean(axis=1), validation_scores.mean(axis=1), strict=True
        ):
            rows.append(
                {
                    "Model": model_name,
                    "Training sessions": int(size),
                    "Training F1": train_score,
                    "CV F1": validation_score,
                }
            )

    axes[0].set_ylabel("F1")
    axes[0].set_ylim(0, 1.05)
    figure.suptitle("Learning curves: does more data help?", fontsize=13)
    figure.tight_layout()
    save_figure(figure, "learning_curves.png")

    table = pd.DataFrame(rows).round(3)
    table.to_csv(os.path.join(OUTPUTS_FOLDER, "learning_curves.csv"), index=False)
    return table


def plot_validation_curves(models, train_features, train_target):
    """
    Draw validation curves: Decision Tree F1 against max_depth and KNN F1
    against n_neighbors, other settings at their tuned values.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        train_features, train_target: the training data.
    Returns:
        A DataFrame with the mean scores, also saved to outputs/validation_curves.csv.
    """
    print("\nValidation curves (F1, 5-fold CV)...")
    settings = [
        (DECISION_TREE, "model__max_depth", TREE_DEPTHS_TO_TRY, "max_depth"),
        (KNN, "model__n_neighbors", NEIGHBOURS_TO_TRY, "n_neighbors (k)"),
    ]
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    rows = []
    for axis, (model_name, parameter, values, axis_label) in zip(axes, settings, strict=True):
        train_scores, validation_scores = validation_curve(
            models[model_name],
            train_features,
            train_target,
            param_name=parameter,
            param_range=values,
            cv=make_folds(),
            scoring="f1",
            n_jobs=-1,
        )
        color = MODEL_COLORS[model_name]
        draw_score_band(axis, values, train_scores, color, "--", "Training F1")
        draw_score_band(axis, values, validation_scores, color, "-", "Cross-validated F1")

        tuned_value = models[model_name].get_params()[parameter]
        axis.axvline(tuned_value, color="grey", linestyle=":", label=f"Tuned value ({tuned_value})")
        axis.set_title(model_name, fontsize=11)
        axis.set_xlabel(axis_label)
        axis.legend(fontsize=9, loc="lower right")
        for value, train_score, validation_score in zip(
            values, train_scores.mean(axis=1), validation_scores.mean(axis=1), strict=True
        ):
            rows.append(
                {
                    "Model": model_name,
                    "Setting": axis_label,
                    "Value": value,
                    "Training F1": train_score,
                    "CV F1": validation_score,
                }
            )

    axes[0].set_ylabel("F1")
    axes[0].set_ylim(0, 1.05)
    figure.suptitle("Validation curves: from underfitting to overfitting", fontsize=13)
    figure.tight_layout()
    save_figure(figure, "validation_curves.png")

    table = pd.DataFrame(rows).round(3)
    table.to_csv(os.path.join(OUTPUTS_FOLDER, "validation_curves.csv"), index=False)
    return table


In [ ]:
learning_table = plot_learning_curves(models, train_features, train_target)
validation_table = plot_validation_curves(models, train_features, train_target)
show("outputs/figures/learning_curves.png")
show("outputs/figures/validation_curves.png")

- The **Decision Tree** has a small gap between training F1 (0.68) and cross-validated F1 (0.64), so its depth limit and minimum leaf size keep overfitting in check, and its cross-validated F1 levels off after about 5,000 sessions.
- **Surprise in the validation curve:** a depth-1 tree, the single rule `PageValues > 0.05`, has the highest cross-validated F1 (0.667, against 0.643 at the tuned depth 7). The grid started at depth 3, so the search never saw it. The deeper tree is still the better model for this project: a one-split tree gives only two probabilities, so it cannot rank sessions (ROC-AUC) or support a profit threshold. Above depth 7, F1 falls as the tree memorises noise. The result is another sign of how much the task rests on PageValues.
- **KNN** scores a perfect training F1 at every size, because distance weighting gives each training session full weight on itself: a textbook case of high variance. Its cross-validated F1 is still rising at the full training size (0.38 to 0.49), so more data would help it. The validation curve peaks at k = 3 and declines as larger neighbourhoods outvote the minority class.

## 9. Which features drive the predictions?

Three views: the Decision Tree's own impurity-based importance, a picture of its top levels, and **permutation importance** for every model. Permutation importance shuffles one original column of the test set and measures how much ROC-AUC drops, so it works for any model and compares them on the same 17 features.

<details><summary>The code below is <code>predator/explain.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
explain.py - Which features drive the predictions?

Three ways of answering that question:
1. Decision Tree feature importance: how much each feature reduced the
   impurity (Gini/entropy) across all the splits that used it.
   Only works for tree models.
2. Permutation importance: shuffle one column of the test set and measure
   how much the model's score drops. If the score drops a lot, the model
   relied on that column. Works for ANY model, so we use it to compare
   all of them on the same 17 original features.
3. Ablation: remove a feature completely (PageValues), retrain every
   model and see how much worse they get.
"""

import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.tree import plot_tree


ABLATION_FEATURE = "PageValues"
PARTIAL_DEPENDENCE_FEATURES = ["PageValues", "Month", "ExitRates"]
PARTIAL_DEPENDENCE_SAMPLE_SIZE = 2000


def plot_tree_feature_importance(tree_pipeline, top_n=15):
    """
    Bar chart of the Decision Tree's built-in feature importance.

    Each value is the total impurity decrease the feature produced, as a
    share of all decreases (the values add up to 1). One-hot columns are
    shown separately, e.g. "Month_Nov".

    Inputs:
        tree_pipeline: the fitted Decision Tree pipeline.
        top_n: how many features to show.
    Returns:
        A DataFrame of all features and their importance, sorted, also saved
        to outputs/tree_feature_importance.csv.
    """
    feature_names = get_feature_names(tree_pipeline)
    importances = tree_pipeline.named_steps["model"].feature_importances_

    importance_table = pd.DataFrame({"feature": feature_names, "importance": importances})
    importance_table = importance_table.sort_values("importance", ascending=False)
    importance_table.to_csv(os.path.join(OUTPUTS_FOLDER, "tree_feature_importance.csv"), index=False)

    top_features = importance_table.head(top_n).iloc[::-1]  # reversed so the biggest is on top
    figure, axis = plt.subplots(figsize=(8, 6))
    axis.barh(top_features["feature"], top_features["importance"], color=MODEL_COLORS[DECISION_TREE])
    axis.set_xlabel("Share of total impurity decrease")
    axis.set_title(f"Decision Tree feature importance (top {top_n})")
    save_figure(figure, "tree_feature_importance.png")
    return importance_table


def plot_top_of_tree(tree_pipeline, max_depth=3):
    """
    Draw the first few levels of the trained Decision Tree.

    The real tree is deeper, but the top levels hold the most important
    questions and are readable. Each box shows the split rule, the
    impurity, how many training sessions reached it, and the majority class.
    Because of class_weight="balanced", the "value" numbers are weighted.

    Inputs:
        tree_pipeline: the fitted Decision Tree pipeline.
        max_depth: how many levels to draw.
    Returns:
        The path of the saved plot.
    """
    tree_model = tree_pipeline.named_steps["model"]
    feature_names = get_feature_names(tree_pipeline)

    figure, axis = plt.subplots(figsize=(30, 11))
    plot_tree(
        tree_model,
        max_depth=max_depth,
        feature_names=feature_names,
        class_names=["No purchase", "Purchase"],
        filled=True,
        rounded=True,
        impurity=True,
        proportion=False,
        precision=2,
        fontsize=10,
        ax=axis,
    )
    axis.set_title(
        f"Top {max_depth} levels of the trained Decision Tree (full depth: {tree_model.get_depth()}, leaves: {tree_model.get_n_leaves()})",
        fontsize=14,
    )
    return save_figure(figure, "decision_tree_top_levels.png")


def compute_permutation_importance(pipeline, test_features, test_target, repeats=5):
    """
    Permutation importance of each ORIGINAL feature for one model.

    We shuffle one column of the test set at a time (breaking its link to
    the target) and measure the drop in ROC-AUC. We shuffle the raw
    columns before preprocessing, so a categorical feature like Month is
    shuffled as one feature, not as ten one-hot columns. This lets us
    compare every model on the same 17 features.

    ROC-AUC is used as the score because it does not depend on the 0.5
    threshold, which suits some models better than others.

    Inputs:
        pipeline: a fitted pipeline.
        test_features, test_target: the test data.
        repeats: how many times each column is shuffled (results are averaged).
    Returns:
        A pandas Series: feature name -> mean drop in ROC-AUC.
    """
    result = permutation_importance(
        pipeline,
        test_features,
        test_target,
        scoring="roc_auc",
        n_repeats=repeats,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return pd.Series(result.importances_mean, index=test_features.columns)


def permutation_importance_all_models(models, test_features, test_target, filename_prefix=""):
    """
    Compute permutation importance for every model and compare them.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        test_features, test_target: the test data.
        filename_prefix: added to output file names (used by the ablation).
    Returns:
        A DataFrame with features as rows and models as columns, sorted
        by the average importance. Saved as a CSV and a heatmap.
    """
    print("\nComputing permutation importance (this takes a minute)...")
    importance_columns = {}
    for model_name, pipeline in models.items():
        print("  " + model_name)
        importance_columns[model_name] = compute_permutation_importance(pipeline, test_features, test_target)

    importance_table = pd.DataFrame(importance_columns)
    importance_table["average"] = importance_table.mean(axis=1)
    importance_table = importance_table.sort_values("average", ascending=False)
    importance_table = importance_table.drop(columns="average")

    importance_table.round(4).to_csv(
        os.path.join(OUTPUTS_FOLDER, filename_prefix + "permutation_importance.csv"),
        index_label="feature",
    )

    figure, axis = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        importance_table, annot=True, fmt=".3f", cmap="Greens", cbar_kws={"label": "Drop in ROC-AUC"}, ax=axis
    )
    title = "Permutation importance: drop in test ROC-AUC when a feature is shuffled"
    if filename_prefix != "":
        title = title + "\n(models retrained without " + ABLATION_FEATURE + ")"
    axis.set_title(title)
    axis.set_ylabel("")
    axis.tick_params(axis="x", rotation=20)
    save_figure(figure, filename_prefix + "permutation_importance.png")
    return importance_table


def rank_features(importance_table):
    """
    Turn an importance table into ranks (1 = most important) per model.

    Input:
        importance_table: features x models DataFrame of importances.
    Returns:
        A DataFrame of the same shape with integer ranks.
    """
    return importance_table.rank(ascending=False, method="min").astype(int)


def run_ablation(
    train_features, train_target, test_features, test_target, full_comparison_table, full_importance_table
):
    """
    Ablation experiment: retrain every model WITHOUT PageValues.

    PageValues is the average value of the pages a visitor saw before
    reaching the page, based on past purchases on the site. It is the
    strongest single signal, so we want to know: how much do the models
    depend on it, and what do they use instead when it is gone?

    Steps:
        1. Drop PageValues from the training and test data.
        2. Tune and train all models again (same grids, same folds).
        3. Compare test scores and feature rankings with the full models.

    Inputs:
        train_features, train_target, test_features, test_target: the data.
        full_comparison_table: test scores of the full models.
        full_importance_table: permutation importance of the full models.
    Returns:
        A dictionary with "metrics" (score comparison), "importance"
        (importance without PageValues) and "top_features" (rank comparison).
    """
    print("\n" + "=" * 60)
    print("ABLATION: retraining every model without " + ABLATION_FEATURE)
    print("=" * 60)

    reduced_numerical = [column for column in NUMERICAL_FEATURES if column != ABLATION_FEATURE]
    reduced_train_features = train_features.drop(columns=ABLATION_FEATURE)
    reduced_test_features = test_features.drop(columns=ABLATION_FEATURE)

    reduced_results = train_all_models(
        reduced_train_features, train_target, reduced_numerical, CATEGORICAL_FEATURES
    )
    reduced_models = {name: result["pipeline"] for name, result in reduced_results.items()}

    reduced_table = build_comparison_table(
        reduced_models, reduced_results, reduced_test_features, test_target, file_prefix="ablation_"
    )

    plot_roc_curves(
        reduced_models,
        reduced_test_features,
        test_target,
        filename="ablation_roc_curves.png",
        title="ROC curves without " + ABLATION_FEATURE,
    )

    # Side-by-side score comparison.
    metrics_table = pd.DataFrame(
        {
            "Model": full_comparison_table["Model"],
            "F1 (all features)": full_comparison_table["F1"].values,
            "F1 (no PageValues)": reduced_table["F1"].values,
            "ROC-AUC (all features)": full_comparison_table["ROC-AUC"].values,
            "ROC-AUC (no PageValues)": reduced_table["ROC-AUC"].values,
        }
    )
    metrics_table["F1 change"] = metrics_table["F1 (no PageValues)"] - metrics_table["F1 (all features)"]
    metrics_table["ROC-AUC change"] = (
        metrics_table["ROC-AUC (no PageValues)"] - metrics_table["ROC-AUC (all features)"]
    )
    metrics_table = metrics_table.round(3)
    metrics_table.to_csv(os.path.join(OUTPUTS_FOLDER, "ablation_metrics.csv"), index=False)
    print("\nAblation results (test set):")
    print(metrics_table.to_string(index=False))
    plot_ablation_scores(metrics_table)

    reduced_importance = permutation_importance_all_models(
        reduced_models, reduced_test_features, test_target, filename_prefix="ablation_"
    )

    # Top 5 features of each model, with and without PageValues.
    full_ranks = rank_features(full_importance_table)
    rows = []
    for model_name in full_importance_table.columns:
        top_full = list(full_ranks[model_name].sort_values().index[:5])
        top_reduced = list(reduced_importance[model_name].sort_values(ascending=False).index[:5])
        rows.append(
            {
                "Model": model_name,
                "Top 5 with all features": ", ".join(top_full),
                "Top 5 without PageValues": ", ".join(top_reduced),
            }
        )
    top_features_table = pd.DataFrame(rows)
    top_features_table.to_csv(os.path.join(OUTPUTS_FOLDER, "ablation_top_features.csv"), index=False)
    print("\nFeature rankings with and without PageValues:")
    print(top_features_table.to_string(index=False))

    return {"metrics": metrics_table, "importance": reduced_importance, "top_features": top_features_table}


def plot_ablation_scores(metrics_table):
    """
    Bar chart of F1 and ROC-AUC with and without PageValues for each model.

    Input:
        metrics_table: the "metrics" table built in run_ablation().
    Returns:
        The path of the saved plot.
    """
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))
    positions = list(range(len(metrics_table)))
    bar_width = 0.38

    for axis, metric in zip(axes, ["F1", "ROC-AUC"], strict=True):
        with_values = metrics_table[metric + " (all features)"]
        without_values = metrics_table[metric + " (no PageValues)"]
        axis.bar(
            [p - bar_width / 2 for p in positions],
            with_values,
            width=bar_width,
            label="All features",
            color="#264653",
        )
        axis.bar(
            [p + bar_width / 2 for p in positions],
            without_values,
            width=bar_width,
            label="Without PageValues",
            color="#E9C46A",
        )
        axis.set_xticks(positions)
        axis.set_xticklabels(metrics_table["Model"], rotation=20, ha="right")
        axis.set_ylim(0, 1)
        axis.set_title(metric + " on the test set")
        axis.legend()

    figure.suptitle("Ablation: what happens when PageValues is removed")
    figure.tight_layout()
    return save_figure(figure, "ablation_scores.png")


def write_tree_vs_xgboost_comparison(comparison_table, models, importance_table):
    """
    Write a short comparison of the Decision Tree and XGBoost: performance
    versus interpretability. Numbers come from this run.

    Inputs:
        comparison_table: test-set scores of all models.
        models: dictionary model name -> fitted pipeline.
        importance_table: permutation importance of all models.
    Returns:
        The text (markdown), also saved to outputs/tree_vs_xgboost.md.
    """
    tree_row = comparison_table[comparison_table["Model"] == DECISION_TREE].iloc[0]
    boost_row = comparison_table[comparison_table["Model"] == XGBOOST].iloc[0]

    tree_model = models[DECISION_TREE].named_steps["model"]
    boost_model = models[XGBOOST].named_steps["model"]

    tree_top = list(importance_table[DECISION_TREE].sort_values(ascending=False).index[:3])
    boost_top = list(importance_table[XGBOOST].sort_values(ascending=False).index[:3])

    text = (
        "# Decision Tree vs XGBoost (benchmark): performance vs interpretability\n\n"
        "| | Decision Tree | XGBoost (benchmark) |\n"
        "|---|---|---|\n"
        "| Test F1 | {tree_f1:.3f} | {boost_f1:.3f} |\n"
        "| Test ROC-AUC | {tree_auc:.3f} | {boost_auc:.3f} |\n"
        "| Precision / Recall | {tree_p:.3f} / {tree_r:.3f} | {boost_p:.3f} / {boost_r:.3f} |\n"
        "| Size | 1 tree, depth {depth}, {leaves} leaves | {trees} trees, depth up to {boost_depth} |\n"
        "| Top 3 features (permutation) | {tree_top} | {boost_top} |\n\n"
        "**Performance.** XGBoost scores {auc_gap:+.3f} ROC-AUC and {f1_gap:+.3f} F1 compared with the "
        "Decision Tree. Boosting combines many small trees, each one correcting the errors of the "
        "previous ones, so it ranks sessions more smoothly than a single tree, whose probabilities "
        "jump in steps (one value per leaf).\n\n"
        "**Interpretability.** The Decision Tree can be read as a flowchart: every prediction follows "
        "one path of yes/no questions, which we can print for any session (the Streamlit app does "
        "this). XGBoost adds up {trees} trees, so no single path explains a prediction; we need "
        "outside tools such as permutation importance to see what it relies on.\n\n"
        "**Agreement.** Both models rely on the same main signal: the top features are "
        "{tree_top} for the tree and {boost_top} for XGBoost.\n\n"
        "**Verdict.** If the small gain in score matters more than being able to explain each "
        "decision, XGBoost is the stronger benchmark. For a shop team that needs to understand and "
        "trust the rules, the Decision Tree gives most of the performance with full transparency.\n"
    ).format(
        tree_f1=tree_row["F1"],
        boost_f1=boost_row["F1"],
        tree_auc=tree_row["ROC-AUC"],
        boost_auc=boost_row["ROC-AUC"],
        tree_p=tree_row["Precision"],
        tree_r=tree_row["Recall"],
        boost_p=boost_row["Precision"],
        boost_r=boost_row["Recall"],
        depth=tree_model.get_depth(),
        leaves=tree_model.get_n_leaves(),
        trees=boost_model.n_estimators,
        boost_depth=boost_model.max_depth,
        tree_top=", ".join(tree_top),
        boost_top=", ".join(boost_top),
        auc_gap=boost_row["ROC-AUC"] - tree_row["ROC-AUC"],
        f1_gap=boost_row["F1"] - tree_row["F1"],
    )

    with open(os.path.join(OUTPUTS_FOLDER, "tree_vs_xgboost.md"), "w") as file:
        file.write(text)
    print("\n" + text)
    return text


def describe_decision_path(tree_pipeline, session):
    """
    List the yes/no questions one session answered on its way down the
    Decision Tree, in plain words.

    Example output line: "PageValues is 204.01, which is > 0.05"
    For one-hot columns (threshold 0.5) it says "Month is Nov" or
    "Month is not Nov" instead of comparing 0/1 numbers.

    Inputs:
        tree_pipeline: the fitted Decision Tree pipeline.
        session: a one-row DataFrame of raw feature values.
    Returns:
        A list of strings, one per question, from the root to the leaf.
    """
    tree_model = tree_pipeline.named_steps["model"]
    feature_names = get_feature_names(tree_pipeline)
    transformed_session = tree_pipeline.named_steps["preprocess"].transform(session)

    # decision_path() marks every node the session visits; indices[...]
    # gives those node numbers in order from the root.
    node_indicator = tree_model.decision_path(transformed_session)
    visited_nodes = node_indicator.indices[node_indicator.indptr[0] : node_indicator.indptr[1]]

    rules = []
    for node in visited_nodes:
        feature_index = tree_model.tree_.feature[node]
        if feature_index < 0:
            continue  # a leaf: no question is asked here
        feature_name = feature_names[feature_index]
        split_value = tree_model.tree_.threshold[node]
        session_value = transformed_session[0, feature_index]
        goes_left = session_value <= split_value

        # Is this a one-hot column like "Month_Nov"?
        original_feature = None
        for categorical_feature in CATEGORICAL_FEATURES:
            if feature_name.startswith(categorical_feature + "_"):
                original_feature = categorical_feature
        if original_feature is not None:
            category = feature_name[len(original_feature) + 1 :]
            if goes_left:
                rules.append(f"{original_feature} is not {category}")
            else:
                rules.append(f"{original_feature} is {category}")
        else:
            if goes_left:
                rules.append(f"{feature_name} is {session_value:.2f}, which is <= {split_value:.2f}")
            else:
                rules.append(f"{feature_name} is {session_value:.2f}, which is > {split_value:.2f}")
    return rules


def plot_partial_dependence(models, model_names, train_features):
    """
    Partial dependence: how the predicted purchase probability changes
    when ONE feature changes and everything else stays as it was.

    For every value on the x axis (e.g. PageValues = 20), the feature is
    set to that value for all sessions in a sample, the model predicts,
    and the predictions are averaged. Permutation importance says HOW MUCH
    a feature matters; partial dependence shows in WHICH DIRECTION and
    with what shape (a jump, a steady rise, a plateau).

    Inputs:
        models: dictionary model name -> fitted pipeline.
        model_names: the models to plot (one row each).
        train_features: the training data (a random sample is used).
    Returns:
        The path of the saved plot.
    """
    print("\nPartial dependence plots for " + ", ".join(PARTIAL_DEPENDENCE_FEATURES) + "...")
    sample = train_features.sample(
        n=min(PARTIAL_DEPENDENCE_SAMPLE_SIZE, len(train_features)), random_state=RANDOM_STATE
    )
    categorical = [feature for feature in PARTIAL_DEPENDENCE_FEATURES if feature in CATEGORICAL_FEATURES]

    figure, axes = plt.subplots(
        len(model_names),
        len(PARTIAL_DEPENDENCE_FEATURES),
        figsize=(5 * len(PARTIAL_DEPENDENCE_FEATURES), 4 * len(model_names)),
    )
    for row, model_name in enumerate(model_names):
        PartialDependenceDisplay.from_estimator(
            models[model_name],
            sample,
            features=PARTIAL_DEPENDENCE_FEATURES,
            categorical_features=categorical,
            response_method="predict_proba",
            ax=axes[row],
            line_kw={"color": MODEL_COLORS[model_name]},
        )
        # Categorical features are drawn as bars; give them the model's colour too.
        for axis in axes[row]:
            for bar in axis.patches:
                bar.set_color(MODEL_COLORS[model_name])
        axes[row][0].set_ylabel(model_name + "\npredicted probability")
        for column in range(1, len(PARTIAL_DEPENDENCE_FEATURES)):
            axes[row][column].set_ylabel("")

    figure.suptitle("Partial dependence: average predicted purchase probability", fontsize=13)
    figure.tight_layout()
    return save_figure(figure, "partial_dependence.png")


In [ ]:
tree_importance = plot_tree_feature_importance(models[DECISION_TREE])
show("outputs/figures/tree_feature_importance.png", width=650)
show(plot_top_of_tree(models[DECISION_TREE], max_depth=3))
importance_table = permutation_importance_all_models(models, test_features, test_target)
show("outputs/figures/permutation_importance.png", width=750)
importance_table.round(3)

The tree's very first question is about **PageValues**, and permutation importance agrees across models: shuffling PageValues costs far more ROC-AUC than any other feature, with **Month** (November) second. Behaviour features like ExitRates and product-page time matter much less.

### Decision Tree vs XGBoost: performance versus interpretability

In [ ]:
display(Markdown(write_tree_vs_xgboost_comparison(comparison_table, models, importance_table)))

### Partial dependence: in which direction does a feature push?

Permutation importance says how much a feature matters. **Partial dependence** shows how: set the feature to each value for 2,000 training sessions, predict, and average. The Decision Tree jumps from about 0.15 to 0.8 as soon as PageValues is above zero; XGBoost climbs more smoothly and levels off around 0.85 by PageValues 25. November raises the probability for both models, and a higher exit rate lowers it.

In [ ]:
show(plot_partial_dependence(models, [DECISION_TREE, XGBOOST], train_features))

## 10. Ablation: what if PageValues did not exist?

PageValues is the average value of the pages a session visited, where a page's value comes from the purchases that followed views of it. If those values were computed over the same period, a buying session's own purchase raised its PageValues: a possible **target leak**. The dataset does not say which period was used, so we measure the dependence instead: drop PageValues, retrain every model with the same grids and folds, and compare scores and feature rankings. The scores without it are the honest lower bound for a deployment that can only use page values from earlier periods.

In [ ]:
ablation = run_ablation(train_features, train_target, test_features, test_target,
                        comparison_table, importance_table)
show("outputs/figures/ablation_scores.png")
show("outputs/figures/ablation_permutation_importance.png", width=750)
ablation["metrics"]

Without PageValues the Decision Tree, KNN and XGBoost lose roughly a third or more of their F1 (Naive Bayes barely changes, because it was already flagging almost every session), and the rankings reshuffle: the models fall back on **Month**, **ExitRates/BounceRates** and **product-page activity**. Those behaviour signals carry some information, but none replaces PageValues.

## 11. Choosing the decision threshold with a business goal

The default 0.5 cutoff is arbitrary. Suppose the shop sends a discount offer to every session the model flags. Using the illustrative assumptions at the top of the next cell (order profit, offer cost, and the share of a buyer's profit the offer secures):

- **true positive**: secured profit minus the offer cost
- **false positive**: the offer cost is wasted
- **false negative**: a missed sale (no extra profit)

We sweep thresholds from 0.01 to 0.99 on the out-of-fold training predictions, pick the best-F1 and best-profit thresholds, and compare them with 0.5 on the test set.

<details><summary>The code below is <code>predator/threshold.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
threshold.py - Choosing the decision threshold with a business goal in mind.

A classifier gives a probability, e.g. "this session has a 0.37 chance of
buying". The decision threshold turns it into a yes/no: flag the session
as a buyer if the probability is at least the threshold. The default 0.5
is just a convention. A lower threshold flags more sessions (higher
recall, lower precision); a higher one flags fewer (the opposite).

The scenario: the shop shows a discount offer to every session the model
flags as a likely buyer. We pick the threshold that earns the most profit
under the assumptions below, and compare it with 0.5 and with the
threshold that gives the best F1 score.

To avoid tuning on the test set, thresholds are chosen on the
out-of-fold training predictions (see train.out_of_fold_probabilities)
and only then applied to the test set.
"""

import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import f1_score, precision_score, recall_score


# ---------------------------------------------------------------------------
# BUSINESS ASSUMPTIONS
# These numbers are ILLUSTRATIVE ASSUMPTIONS, not facts from the dataset.
# Change them to match a real shop and re-run main.py; the best threshold
# will move accordingly.
# ---------------------------------------------------------------------------

# Profit (not revenue) the shop makes on one average order, in dollars.
AVERAGE_ORDER_PROFIT = 60.0

# What one offer costs the shop (the discount given plus delivery cost),
# in dollars. Paid for every session we target, buyer or not.
OFFER_COST = 6.0

# Share of a targeted buyer's order profit that we credit to the offer:
# the fraction of these sales the offer actually wins or saves (the rest
# would have happened anyway). 0.40 means 40%.
OFFER_CONVERSION_BOOST = 0.40

# Thresholds we try, from 0.01 to 0.99.
THRESHOLDS_TO_TRY = np.round(np.arange(0.01, 1.00, 0.01), 2)


def profit_per_outcome(
    order_profit=AVERAGE_ORDER_PROFIT, offer_cost=OFFER_COST, conversion_boost=OFFER_CONVERSION_BOOST
):
    """
    The profit from one session for each of the four possible outcomes.

    The three inputs default to the business assumptions above; the
    sensitivity analysis and the app pass other values.

    - True positive (flagged, really a buyer): profit from the sale the
      offer secures, minus the offer cost.
    - False positive (flagged, not a buyer): the offer cost is wasted.
    - False negative (not flagged, really a buyer): a missed sale. We earn
      nothing extra from the campaign; the lost chance is reported
      separately as "missed profit".
    - True negative (not flagged, not a buyer): nothing happens.

    Returns:
        A dictionary with the profit for "tp", "fp", "fn" and "tn".
    """
    return {
        "tp": order_profit * conversion_boost - offer_cost,
        "fp": -offer_cost,
        "fn": 0.0,
        "tn": 0.0,
    }


def calculate_profit(
    true_target,
    probabilities,
    threshold,
    order_profit=AVERAGE_ORDER_PROFIT,
    offer_cost=OFFER_COST,
    conversion_boost=OFFER_CONVERSION_BOOST,
):
    """
    Work out the campaign profit if we target every session whose
    probability is at least the threshold.

    Inputs:
        true_target: the real labels (0/1).
        probabilities: the model's purchase probabilities.
        threshold: the probability cutoff.
        order_profit, offer_cost, conversion_boost: the business
            assumptions (default: the constants at the top of this file).
    Returns:
        A dictionary with the total "profit", the counts "tp", "fp", "fn",
        "tn", the number of "targeted" sessions and the "missed_profit"
        from buyers we did not target.
    """
    true_target = np.asarray(true_target)
    predictions = (np.asarray(probabilities) >= threshold).astype(int)

    true_positives = int(((predictions == 1) & (true_target == 1)).sum())
    false_positives = int(((predictions == 1) & (true_target == 0)).sum())
    false_negatives = int(((predictions == 0) & (true_target == 1)).sum())
    true_negatives = int(((predictions == 0) & (true_target == 0)).sum())

    values = profit_per_outcome(order_profit, offer_cost, conversion_boost)
    total_profit = (
        true_positives * values["tp"]
        + false_positives * values["fp"]
        + false_negatives * values["fn"]
        + true_negatives * values["tn"]
    )
    missed_profit = false_negatives * values["tp"]

    return {
        "profit": float(total_profit),
        "tp": true_positives,
        "fp": false_positives,
        "fn": false_negatives,
        "tn": true_negatives,
        "targeted": true_positives + false_positives,
        "missed_profit": float(missed_profit),
    }


def build_threshold_curve(true_target, probabilities):
    """
    Try every threshold in THRESHOLDS_TO_TRY and record profit, precision,
    recall and F1 at each one.

    Inputs:
        true_target: the real labels (0/1).
        probabilities: the model's purchase probabilities.
    Returns:
        A DataFrame with one row per threshold.
    """
    rows = []
    for threshold in THRESHOLDS_TO_TRY:
        predictions = (probabilities >= threshold).astype(int)
        profit_result = calculate_profit(true_target, probabilities, threshold)
        rows.append(
            {
                "threshold": threshold,
                "profit": profit_result["profit"],
                "precision": precision_score(true_target, predictions, zero_division=0),
                "recall": recall_score(true_target, predictions, zero_division=0),
                "f1": f1_score(true_target, predictions, zero_division=0),
            }
        )
    return pd.DataFrame(rows)


def find_best_thresholds(curve):
    """
    Find the threshold with the highest F1 and the one with the highest profit.

    If several thresholds tie, the first (lowest) one is used.

    Input:
        curve: the DataFrame from build_threshold_curve().
    Returns:
        (best_f1_threshold, best_profit_threshold) as floats.
    """
    best_f1_threshold = float(curve.loc[curve["f1"].idxmax(), "threshold"])
    best_profit_threshold = float(curve.loc[curve["profit"].idxmax(), "threshold"])
    return best_f1_threshold, best_profit_threshold


def tune_thresholds(models, training_results, train_target, test_features, test_target):
    """
    Choose the best-F1 and best-profit thresholds for every model and
    compare them with the default 0.5 on the test set.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        training_results: from train.train_all_models() (holds the
            out-of-fold training probabilities).
        train_target: training labels.
        test_features, test_target: the test data.
    Returns:
        (comparison_table, chosen_thresholds, training_curves)
        - comparison_table: DataFrame, also saved to outputs/threshold_comparison.csv
        - chosen_thresholds: model name -> {"default", "best_f1", "best_profit"}
        - training_curves: model name -> curve DataFrame (for plotting)
    """
    print("\nTuning decision thresholds (business angle)...")
    values = profit_per_outcome()
    print(
        f"  Assumptions: order profit ${AVERAGE_ORDER_PROFIT:.0f}, offer cost ${OFFER_COST:.0f}, offer conversion boost {OFFER_CONVERSION_BOOST:.0%}"
    )
    print(
        "  Profit per true positive: ${:.2f}, per false positive: ${:.2f}".format(values["tp"], values["fp"])
    )

    rows = []
    chosen_thresholds = {}
    training_curves = {}

    for model_name, pipeline in models.items():
        training_probabilities = training_results[model_name]["oof_probabilities"]
        curve = build_threshold_curve(train_target, training_probabilities)
        best_f1_threshold, best_profit_threshold = find_best_thresholds(curve)
        training_curves[model_name] = curve
        chosen_thresholds[model_name] = {
            "default": 0.5,
            "best_f1": best_f1_threshold,
            "best_profit": best_profit_threshold,
        }

        test_probabilities = get_purchase_probabilities(pipeline, test_features)
        for label, threshold in [
            ("Default 0.5", 0.5),
            ("Best F1", best_f1_threshold),
            ("Best profit", best_profit_threshold),
        ]:
            predictions = (test_probabilities >= threshold).astype(int)
            profit_result = calculate_profit(test_target, test_probabilities, threshold)
            rows.append(
                {
                    "Model": model_name,
                    "Threshold type": label,
                    "Threshold": threshold,
                    "Precision": precision_score(test_target, predictions, zero_division=0),
                    "Recall": recall_score(test_target, predictions, zero_division=0),
                    "F1": f1_score(test_target, predictions, zero_division=0),
                    "Sessions targeted": profit_result["targeted"],
                    "Profit ($)": profit_result["profit"],
                }
            )

    comparison_table = pd.DataFrame(rows).round(3)
    comparison_table.to_csv(os.path.join(OUTPUTS_FOLDER, "threshold_comparison.csv"), index=False)
    print(f"\nThreshold comparison on the test set ({len(test_target)} sessions):")
    print(comparison_table.to_string(index=False))
    return comparison_table, chosen_thresholds, training_curves


def plot_profit_curves(training_curves, chosen_thresholds):
    """
    Plot profit and F1 against the threshold for every model.

    One small chart per model. The solid line is profit (left axis), the
    dotted line is F1 (right axis). Vertical lines mark the best-profit
    threshold, the best-F1 threshold and the default 0.5.
    The curves come from the out-of-fold training predictions, which is
    where the thresholds were chosen.

    Inputs:
        training_curves: model name -> curve DataFrame.
        chosen_thresholds: model name -> chosen thresholds.
    Returns:
        The path of the saved plot.
    """
    model_names = list(training_curves.keys())
    figure, axes = plt.subplots(1, len(model_names), figsize=(4.6 * len(model_names), 4.4), sharey=True)

    for axis, model_name in zip(axes, model_names, strict=True):
        curve = training_curves[model_name]
        thresholds = chosen_thresholds[model_name]
        color = MODEL_COLORS[model_name]

        axis.plot(curve["threshold"], curve["profit"], color=color, linewidth=2, label="Profit")
        axis.axhline(0, color="grey", linewidth=0.8)
        axis.axvline(
            thresholds["best_profit"],
            color=color,
            linestyle="--",
            label="Best profit ({:.2f})".format(thresholds["best_profit"]),
        )
        axis.axvline(
            thresholds["best_f1"],
            color="black",
            linestyle=":",
            label="Best F1 ({:.2f})".format(thresholds["best_f1"]),
        )
        axis.axvline(0.5, color="grey", linestyle="-.", linewidth=1, label="Default 0.5")
        axis.set_title(model_name, fontsize=11)
        axis.set_xlabel("Threshold")

        f1_axis = axis.twinx()
        f1_axis.plot(curve["threshold"], curve["f1"], color="black", linestyle=":", linewidth=1.2, alpha=0.6)
        f1_axis.set_ylim(0, 1)
        if model_name == model_names[-1]:
            f1_axis.set_ylabel("F1 (dotted line)")
        else:
            f1_axis.set_yticklabels([])
        axis.legend(fontsize=8, loc="lower left")

    axes[0].set_ylabel("Profit on training data ($, out-of-fold)")
    figure.suptitle("Profit and F1 versus decision threshold", fontsize=13)
    figure.tight_layout()
    return save_figure(figure, "profit_vs_threshold.png")


# ---------------------------------------------------------------------------
# SENSITIVITY ANALYSIS
# The assumptions above are guesses, so we check how the best threshold
# moves when they change. These are the values tried for the two
# assumptions that are hardest to know in advance.
# ---------------------------------------------------------------------------
OFFER_COSTS_TO_TRY = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0]
CONVERSION_BOOSTS_TO_TRY = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]


def break_even_probability(
    order_profit=AVERAGE_ORDER_PROFIT, offer_cost=OFFER_COST, conversion_boost=OFFER_CONVERSION_BOOST
):
    """
    The purchase probability above which sending an offer pays off.

    If a session buys with probability p, sending it the offer earns on
    average:
        p * (order_profit * conversion_boost - offer_cost) - (1 - p) * offer_cost
        = p * order_profit * conversion_boost - offer_cost
    This is above zero when p > offer_cost / (order_profit * conversion_boost).

    That cutoff is only right if the model's probabilities are calibrated
    (a "0.3" really means 30% of such sessions buy); see calibration.py.

    Returns:
        The break-even probability as a float.
    """
    return offer_cost / (order_profit * conversion_boost)


def find_best_profit_threshold(true_target, probabilities, order_profit, offer_cost, conversion_boost):
    """
    Try every threshold in THRESHOLDS_TO_TRY and return the most profitable one.

    Inputs:
        true_target: the real labels (0/1).
        probabilities: the model's purchase probabilities.
        order_profit, offer_cost, conversion_boost: the business assumptions.
    Returns:
        (best_threshold, best_profit). If several thresholds tie, the first
        (lowest) one is used. If no threshold makes a profit, the answer is
        (1.0, 0.0): send no offers at all.
    """
    best_threshold = 1.0
    best_profit = 0.0
    for threshold in THRESHOLDS_TO_TRY:
        profit = calculate_profit(
            true_target, probabilities, threshold, order_profit, offer_cost, conversion_boost
        )["profit"]
        if profit > best_profit:
            best_threshold = float(threshold)
            best_profit = profit
    return best_threshold, best_profit


def run_sensitivity_analysis(training_results, train_target, model_names):
    """
    Find the best-profit threshold for every combination of offer cost and
    conversion boost, for each of the given models.

    Like tune_thresholds(), this uses the out-of-fold training
    probabilities, so the test set is not touched.

    Inputs:
        training_results: from train.train_all_models().
        train_target: the training labels.
        model_names: the models to analyse (e.g. the Decision Tree and XGBoost).
    Returns:
        A DataFrame with one row per model and combination, also saved to
        outputs/sensitivity_analysis.csv.
    """
    print("\nSensitivity analysis: best threshold for other business assumptions...")
    true_target = np.asarray(train_target)
    rows = []
    for model_name in model_names:
        probabilities = training_results[model_name]["oof_probabilities"]
        for offer_cost in OFFER_COSTS_TO_TRY:
            for conversion_boost in CONVERSION_BOOSTS_TO_TRY:
                best_threshold, best_profit = find_best_profit_threshold(
                    true_target, probabilities, AVERAGE_ORDER_PROFIT, offer_cost, conversion_boost
                )
                rows.append(
                    {
                        "Model": model_name,
                        "Offer cost": offer_cost,
                        "Conversion boost": conversion_boost,
                        "Break-even probability": round(
                            break_even_probability(AVERAGE_ORDER_PROFIT, offer_cost, conversion_boost), 3
                        ),
                        "Best threshold": best_threshold,
                        "Training profit": round(best_profit, 2),
                    }
                )

    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(OUTPUTS_FOLDER, "sensitivity_analysis.csv"), index=False)
    plot_sensitivity_heatmaps(table, model_names)
    return table


def plot_sensitivity_heatmaps(sensitivity_table, model_names):
    """
    Draw one heatmap per model: the best threshold for each offer cost
    (rows) and conversion boost (columns).

    Inputs:
        sensitivity_table: the DataFrame from run_sensitivity_analysis().
        model_names: the models in the table.
    Returns:
        The path of the saved plot.
    """
    figure, axes = plt.subplots(1, len(model_names), figsize=(6.2 * len(model_names), 5))
    for axis, model_name in zip(axes, model_names, strict=True):
        rows = sensitivity_table[sensitivity_table["Model"] == model_name]
        grid = rows.pivot(index="Offer cost", columns="Conversion boost", values="Best threshold")
        sns.heatmap(grid, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=axis, cbar=False)
        axis.set_title(model_name, fontsize=11)
        axis.set_xlabel("Conversion boost from the offer (1.00 = send no offers)")
        axis.set_ylabel("Offer cost ($)")
    figure.suptitle(
        "Best-profit threshold for other business assumptions (order profit $"
        + str(int(AVERAGE_ORDER_PROFIT))
        + ")",
        fontsize=13,
    )
    figure.tight_layout()
    return save_figure(figure, "sensitivity_thresholds.png")


In [ ]:
threshold_table, chosen_thresholds, training_curves = tune_thresholds(
    models, training_results, train_target, test_features, test_target)
show(plot_profit_curves(training_curves, chosen_thresholds))
threshold_table

The best threshold depends on the model's probability calibration as much as on the business numbers. The class-weighted Decision Tree, for example, pushes probabilities upwards, so its best-profit threshold sits well above 0.5. Naive Bayes loses money at every threshold, another sign its probabilities cannot be trusted.

### How sensitive is the best threshold to the assumptions?

The offer cost and the conversion boost are guesses, so we repeat the search for 36 combinations of them (still on out-of-fold training data). A threshold of 1.00 means no session is worth an offer.

In [ ]:
sensitivity_table = run_sensitivity_analysis(training_results, train_target, [DECISION_TREE, XGBOOST])
show("outputs/figures/sensitivity_thresholds.png")

Cheaper offers and stronger boosts push the best threshold down (target more sessions); with a 10% boost and an offer cost of $6 or more, no threshold makes a profit. The ordering is stable, so the conclusion "use a cutoff above 0.5 for the tree" holds over a wide range of assumptions, while the exact number does not.

## 12. Probability calibration: can we trust the numbers?

ROC-AUC only checks the **order** of the probabilities. **Calibration** checks the numbers: of the sessions given about 0.3, about 30% should buy. If they are calibrated, the best threshold follows from the assumptions alone: send the offer when p is above offer cost / (order profit x conversion boost), which is **0.25** here.

The **reliability diagram** plots predicted probability against the real purchase rate (the diagonal is perfect), and the **Brier score** is the mean squared error of the probabilities. We then calibrate the Decision Tree and XGBoost with isotonic regression on the training folds.

<details><summary>The code below is <code>predator/calibration.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
calibration.py - Can we trust the probabilities, not just the ranking?

ROC-AUC only checks the ORDER of the probabilities: buyers should get
higher numbers than non-buyers. Calibration checks the numbers
themselves: of all sessions that get a probability of about 0.3, about
30% should really buy.

This matters for the business threshold. The break-even rule in
threshold.break_even_probability() ("send the offer when p is above
cost / (profit x boost)") only works with calibrated probabilities.

Two of our models are miscalibrated on purpose: the Decision Tree uses
class_weight="balanced" and XGBoost uses scale_pos_weight. Both make
buyers count more during training, which pushes every probability up.

We measure calibration with:
- a reliability diagram: predicted probability (x) against the real
  purchase rate (y) in bins; a calibrated model follows the diagonal.
- the Brier score: the mean squared error between the probability and
  the 0/1 outcome (lower is better).

Then we fix it with isotonic regression (CalibratedClassifierCV), which
learns a monotone mapping from the raw probability to the real rate on
cross-validation folds of the training data.
"""

import matplotlib

matplotlib.use("Agg")
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score


NUMBER_OF_BINS = 10


def calibrate_model(pipeline, train_features, train_target):
    """
    Wrap a tuned pipeline in isotonic calibration and fit it.

    CalibratedClassifierCV trains a copy of the pipeline on 4 of the 5
    folds and learns the probability mapping on the fold it did not see,
    five times. Its predictions average the five calibrated copies. Only
    training data is used.

    Inputs:
        pipeline: a fitted pipeline (its settings are copied, not its fit).
        train_features, train_target: the training data.
    Returns:
        The fitted CalibratedClassifierCV.
    """
    calibrated = CalibratedClassifierCV(clone(pipeline), method="isotonic", cv=make_folds())
    calibrated.fit(train_features, train_target)
    return calibrated


def plot_reliability_diagram(probabilities_by_model, test_target, filename, title):
    """
    Draw the reliability diagram for several sets of probabilities.

    Bins hold equal numbers of sessions ("quantile" bins), so every point
    is based on enough sessions.

    Inputs:
        probabilities_by_model: dictionary label -> (probabilities, colour, line style).
        test_target: the real labels.
        filename: the file name in outputs/figures/.
        title: the plot title.
    Returns:
        The path of the saved plot.
    """
    figure, axis = plt.subplots(figsize=(7, 6))
    axis.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1, label="Perfectly calibrated")

    for label, (probabilities, color, line_style) in probabilities_by_model.items():
        real_rate, mean_predicted = calibration_curve(
            test_target, probabilities, n_bins=NUMBER_OF_BINS, strategy="quantile"
        )
        brier = brier_score_loss(test_target, probabilities)
        axis.plot(
            mean_predicted,
            real_rate,
            marker="o",
            color=color,
            linestyle=line_style,
            label=f"{label} (Brier {brier:.3f})",
        )

    axis.set_xlabel("Predicted purchase probability")
    axis.set_ylabel("Real purchase rate in the bin")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.set_title(title, fontsize=12)
    axis.legend(fontsize=8, loc="upper left")
    figure.tight_layout()
    return save_figure(figure, filename)


def run_calibration_analysis(
    models, model_names_to_calibrate, train_features, train_target, test_features, test_target
):
    """
    Measure calibration for every model, calibrate the chosen ones, and
    compare the profit of the break-even rule before and after.

    Inputs:
        models: dictionary model name -> fitted pipeline.
        model_names_to_calibrate: the models to calibrate (the Decision Tree
            and XGBoost: the two whose training reweights buyers).
        train_features, train_target, test_features, test_target: the data.
    Returns:
        A DataFrame with one row per model, also saved to
        outputs/calibration_summary.csv.
    """
    print("\nChecking how well the probabilities are calibrated...")
    cutoff = break_even_probability()
    print(f"  Break-even probability for the business assumptions: {cutoff:.2f}")

    all_models = {}
    for model_name, pipeline in models.items():
        all_models[model_name] = (
            get_purchase_probabilities(pipeline, test_features),
            MODEL_COLORS[model_name],
            "-",
        )
    plot_reliability_diagram(
        all_models, test_target, "calibration_all_models.png", "Reliability diagram (test set)"
    )

    rows = []
    before_and_after = {}
    for model_name, pipeline in models.items():
        raw_probabilities = all_models[model_name][0]
        row = {
            "Model": model_name,
            "Brier (raw)": brier_score_loss(test_target, raw_probabilities),
            "ROC-AUC (raw)": roc_auc_score(test_target, raw_probabilities),
            "Mean probability (raw)": raw_probabilities.mean(),
            "Profit at break-even (raw)": calculate_profit(test_target, raw_probabilities, cutoff)["profit"],
        }

        if model_name in model_names_to_calibrate:
            print("  Calibrating " + model_name + " (isotonic, 5 folds)...")
            calibrated = calibrate_model(pipeline, train_features, train_target)
            calibrated_probabilities = calibrated.predict_proba(test_features)[:, 1]
            row["Brier (calibrated)"] = brier_score_loss(test_target, calibrated_probabilities)
            row["ROC-AUC (calibrated)"] = roc_auc_score(test_target, calibrated_probabilities)
            row["Mean probability (calibrated)"] = calibrated_probabilities.mean()
            row["Profit at break-even (calibrated)"] = calculate_profit(
                test_target, calibrated_probabilities, cutoff
            )["profit"]
            color = MODEL_COLORS[model_name]
            before_and_after[model_name + " raw"] = (raw_probabilities, color, ":")
            before_and_after[model_name + " calibrated"] = (calibrated_probabilities, color, "-")
        rows.append(row)

    plot_reliability_diagram(
        before_and_after,
        test_target,
        "calibration_before_after.png",
        "Before and after isotonic calibration (test set)",
    )

    table = pd.DataFrame(rows).round(3)
    table.to_csv(os.path.join(OUTPUTS_FOLDER, "calibration_summary.csv"), index=False)
    print(f"  Real purchase rate in the test set: {test_target.mean():.3f}")
    print(table.to_string(index=False))
    return table


In [ ]:
calibration_table = run_calibration_analysis(
    models, [DECISION_TREE, XGBOOST], train_features, train_target, test_features, test_target)
show("outputs/figures/calibration_all_models.png", width=600)
show("outputs/figures/calibration_before_after.png", width=600)
calibration_table

The Decision Tree's raw probabilities average 0.28 although only 15.5% of sessions buy: the balanced class weights inflate them. After calibration they average 0.15, the Brier score drops from 0.109 to 0.072, and ROC-AUC is unchanged. With calibrated probabilities the simple break-even rule (p > 0.25) earns **$4,086** on the test set, more than the tuned 0.70 threshold on the raw probabilities ($3,870). XGBoost improves the same way (Brier 0.094 to 0.071, $4,020). Calibration turns the threshold from a tuned number into one you can explain.

## 13. Error analysis: who does the best model get wrong?

We label every test session as a caught buyer, missed buyer, false alarm or correct non-buyer (using the best model at its best-profit threshold) and compare the groups.

<details><summary>The code below is <code>predator/errors.py</code> from the GitHub repo.</summary>Each function has a docstring and comments explaining the method.</details>

In [ ]:
"""
errors.py - Error analysis: which sessions does the best model get wrong, and why?

We split the test sessions into four groups using the best model's
predictions at its chosen threshold:
    - Caught buyers (true positives)
    - Missed buyers (false negatives): bought, but the model said no
    - False alarms (false positives): did not buy, but the model said yes
    - Correct non-buyers (true negatives)
Then we compare the groups' feature averages and distributions to see
what the mistakes have in common.
"""

import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


CAUGHT_BUYER = "Caught buyer (TP)"
MISSED_BUYER = "Missed buyer (FN)"
FALSE_ALARM = "False alarm (FP)"
CORRECT_NON_BUYER = "Correct non-buyer (TN)"
GROUP_ORDER = [CAUGHT_BUYER, MISSED_BUYER, FALSE_ALARM, CORRECT_NON_BUYER]
GROUP_COLORS = {
    CAUGHT_BUYER: "#2A9D8F",
    MISSED_BUYER: "#E76F51",
    FALSE_ALARM: "#E9C46A",
    CORRECT_NON_BUYER: "#9AA5B1",
}
MONTH_ORDER = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def label_prediction_groups(pipeline, test_features, test_target, threshold):
    """
    Put every test session into one of the four outcome groups.

    Inputs:
        pipeline: the fitted best model.
        test_features, test_target: the test data.
        threshold: the decision threshold used for the best model.
    Returns:
        A copy of test_features with extra columns: "actual",
        "probability", "predicted" and "group".
    """
    sessions = test_features.copy()
    sessions["actual"] = test_target.values
    sessions["probability"] = get_purchase_probabilities(pipeline, test_features)
    sessions["predicted"] = (sessions["probability"] >= threshold).astype(int)

    groups = []
    for actual, predicted in zip(sessions["actual"], sessions["predicted"], strict=True):
        if actual == 1 and predicted == 1:
            groups.append(CAUGHT_BUYER)
        elif actual == 1 and predicted == 0:
            groups.append(MISSED_BUYER)
        elif actual == 0 and predicted == 1:
            groups.append(FALSE_ALARM)
        else:
            groups.append(CORRECT_NON_BUYER)
    sessions["group"] = groups
    return sessions


def summarise_numerical_features(sessions):
    """
    Average of every numerical feature for each group, plus a few extras.

    Input:
        sessions: the DataFrame from label_prediction_groups().
    Returns:
        A DataFrame with one row per group, saved to
        outputs/error_numerical_summary.csv.
    """
    summary = sessions.groupby("group")[NUMERICAL_FEATURES + ["probability"]].mean()
    summary["sessions"] = sessions.groupby("group").size()
    summary["share with PageValues = 0"] = sessions.groupby("group")["PageValues"].apply(
        lambda values: (values == 0).mean()
    )
    summary["median ProductRelated_Duration"] = sessions.groupby("group")["ProductRelated_Duration"].median()
    summary = summary.reindex(GROUP_ORDER).round(3)
    summary.to_csv(os.path.join(OUTPUTS_FOLDER, "error_numerical_summary.csv"), index_label="group")
    return summary


def summarise_categorical_feature(sessions, column):
    """
    Share of each category inside every group (each row adds up to 100%).

    For example, for VisitorType: what share of missed buyers were new
    visitors, compared with caught buyers.

    Inputs:
        sessions: the DataFrame from label_prediction_groups().
        column: a categorical column name.
    Returns:
        A DataFrame: groups as rows, categories as columns, values are shares.
    """
    shares = pd.crosstab(sessions["group"], sessions[column], normalize="index")
    shares = shares.reindex(GROUP_ORDER)
    if column == "Month":
        shares = shares[[month for month in MONTH_ORDER if month in shares.columns]]
    return shares.round(3)


def plot_error_analysis(sessions, model_name):
    """
    Four plots comparing the groups: PageValues, ProductRelated_Duration,
    VisitorType and Month.

    PageValues and durations are very skewed, so the box plots use a
    log scale (log(1 + value)) to keep the zeros and the big values visible.

    Inputs:
        sessions: the DataFrame from label_prediction_groups().
        model_name: shown in the title.
    Returns:
        The path of the saved plot.
    """
    figure, axes = plt.subplots(2, 2, figsize=(16, 11))
    colors = [GROUP_COLORS[group] for group in GROUP_ORDER]
    short_labels = [group.replace(" (", "\n(") for group in GROUP_ORDER]

    # 1 and 2: box plots of two numerical features.
    for axis, column in zip([axes[0][0], axes[0][1]], ["PageValues", "ProductRelated_Duration"], strict=True):
        values_per_group = [
            np.log1p(sessions.loc[sessions["group"] == group, column]) for group in GROUP_ORDER
        ]
        boxes = axis.boxplot(values_per_group, patch_artist=True, showfliers=False)
        axis.set_xticks(range(1, len(GROUP_ORDER) + 1))
        axis.set_xticklabels(short_labels)
        for box, color in zip(boxes["boxes"], colors, strict=True):
            box.set_facecolor(color)
        axis.set_title(column + " by outcome group")
        axis.set_ylabel("log(1 + " + column + ")")

    # 3: VisitorType share per group.
    visitor_shares = summarise_categorical_feature(sessions, "VisitorType")
    visitor_shares.plot(kind="bar", ax=axes[1][0], color=["#264653", "#E9C46A", "#E76F51"], rot=0)
    axes[1][0].set_xticklabels(short_labels)
    axes[1][0].set_title("Visitor type within each group")
    axes[1][0].set_ylabel("Share of the group")
    axes[1][0].set_xlabel("")
    axes[1][0].legend(title="VisitorType", loc="upper left")

    # 4: Month share for the three groups that matter most.
    month_shares = summarise_categorical_feature(sessions, "Month")
    for group in [CAUGHT_BUYER, MISSED_BUYER, FALSE_ALARM]:
        axes[1][1].plot(
            month_shares.columns,
            month_shares.loc[group],
            marker="o",
            color=GROUP_COLORS[group],
            linewidth=2,
            label=group,
        )
    axes[1][1].set_title("Month within each group")
    axes[1][1].set_ylabel("Share of the group")
    axes[1][1].legend()

    figure.suptitle("Error analysis of " + model_name + " on the test set", fontsize=14)
    figure.tight_layout()
    return save_figure(figure, "error_analysis.png")


def plot_probability_histogram(sessions, threshold, model_name):
    """
    Histogram of predicted probabilities for real buyers and non-buyers.

    Missed buyers are the real buyers left of the threshold line; false
    alarms are the non-buyers right of it.

    Inputs:
        sessions: the DataFrame from label_prediction_groups().
        threshold: the decision threshold.
        model_name: shown in the title.
    Returns:
        The path of the saved plot.
    """
    figure, axis = plt.subplots(figsize=(9, 5))
    bins = np.linspace(0, 1, 41)
    axis.hist(
        sessions.loc[sessions["actual"] == 0, "probability"],
        bins=bins,
        alpha=0.7,
        color=GROUP_COLORS[CORRECT_NON_BUYER],
        label="Did not buy",
    )
    axis.hist(
        sessions.loc[sessions["actual"] == 1, "probability"],
        bins=bins,
        alpha=0.7,
        color=GROUP_COLORS[CAUGHT_BUYER],
        label="Bought",
    )
    axis.axvline(threshold, color="black", linestyle="--", label=f"Threshold {threshold:.2f}")
    axis.set_yscale("log")
    axis.set_xlabel("Predicted purchase probability")
    axis.set_ylabel("Number of sessions (log scale)")
    axis.set_title("Predicted probabilities of " + model_name)
    axis.legend()
    return save_figure(figure, "error_probability_histogram.png")


def write_error_findings(sessions, numerical_summary, model_name, threshold):
    """
    Write the main error-analysis findings in plain English, using the
    numbers from this run.

    Inputs:
        sessions: the DataFrame from label_prediction_groups().
        numerical_summary: the DataFrame from summarise_numerical_features().
        model_name: the best model's name.
        threshold: its decision threshold.
    Returns:
        The findings text (markdown), also saved to outputs/error_analysis.md.
    """
    visitor_shares = summarise_categorical_feature(sessions, "VisitorType")
    month_shares = summarise_categorical_feature(sessions, "Month")

    caught = numerical_summary.loc[CAUGHT_BUYER]
    missed = numerical_summary.loc[MISSED_BUYER]
    false_alarm = numerical_summary.loc[FALSE_ALARM]

    # The busiest month is always the most common one in every group, so
    # instead we look for the month that is MORE common among the mistakes
    # than among the caught buyers.
    missed_month_gap = month_shares.loc[MISSED_BUYER] - month_shares.loc[CAUGHT_BUYER]
    missed_month = missed_month_gap.idxmax()
    false_alarm_month_gap = month_shares.loc[FALSE_ALARM] - month_shares.loc[CAUGHT_BUYER]
    false_alarm_month = false_alarm_month_gap.idxmax()

    missed_new = visitor_shares.loc[MISSED_BUYER].get("New_Visitor", 0.0)
    caught_new = visitor_shares.loc[CAUGHT_BUYER].get("New_Visitor", 0.0)
    if abs(missed_new - caught_new) < 0.05:
        visitor_sentence = (
            f"**Visitor type is not the reason.** {missed_new:.0%} of missed buyers are new visitors, "
            f"about the same as caught buyers ({caught_new:.0%})."
        )
    else:
        visitor_sentence = (
            f"**Visitor type.** {missed_new:.0%} of missed buyers are new visitors, against {caught_new:.0%} "
            "of caught buyers."
        )

    text = (
        "# Error analysis: "
        + model_name
        + f" (threshold {threshold:.2f})\n\n"
        + "| Group | Sessions | Mean PageValues | Share with PageValues = 0 | Median product time (s) "
        "| Mean ExitRates | Mean probability |\n" + "|---|---|---|---|---|---|---|\n"
    )
    for group in GROUP_ORDER:
        row = numerical_summary.loc[group]
        text = text + "| {} | {:.0f} | {:.1f} | {:.0%} | {:.0f} | {:.3f} | {:.2f} |\n".format(
            group,
            row["sessions"],
            row["PageValues"],
            row["share with PageValues = 0"],
            row["median ProductRelated_Duration"],
            row["ExitRates"],
            row["probability"],
        )

    text = text + (
        "\n## What the missed buyers have in common\n\n"
        "- **They look like browsers, not buyers.** {missed_zero:.0%} of missed buyers have "
        "PageValues = 0, against {caught_zero:.0%} of caught buyers. PageValues is the model's "
        "strongest signal, so a buyer who never passed through a page linked to earlier purchases "
        "gets a low probability (mean {missed_prob:.2f}).\n"
        "- **Less engagement.** Their median time on product pages is {missed_time:.0f} s, against "
        "{caught_time:.0f} s for caught buyers, and their mean exit rate is higher "
        "({missed_exit:.3f} vs {caught_exit:.3f}).\n"
        "- {visitor_sentence}\n"
        "- **Timing.** {missed_month} is over-represented: {missed_month_share:.0%} of missed buyers "
        "against {missed_month_caught:.0%} of caught buyers.\n\n"
        "## What the false alarms have in common\n\n"
        "- They look like buyers: mean PageValues {fa_pv:.1f} (caught buyers: {caught_pv:.1f}) "
        "and only {fa_zero:.0%} have PageValues = 0. They reached valuable pages and then left "
        "without paying, which the features cannot tell apart from a purchase.\n"
        "- {fa_returning:.0%} of false alarms are returning visitors, against {caught_returning:.0%} "
        "of caught buyers: people who come back, browse valuable pages and still do not buy.\n"
        "- {fa_month} is over-represented: {fa_month_share:.0%} of false alarms against "
        "{fa_month_caught:.0%} of caught buyers.\n\n"
        "## Why the model struggles\n\n"
        "The dataset records how a session behaved, not why. A buyer who already knew what to buy "
        "can arrive, buy quickly and leave, without the page-value trail that most buyers leave. "
        "And a visitor who filled a cart and abandoned it leaves exactly that trail. With "
        "PageValues as the dominant feature, the model's mistakes are mostly the sessions where "
        "PageValues points the wrong way. More data about the visitor (past orders, cart "
        "contents) would be needed to fix them; tuning the model further cannot.\n"
    ).format(
        missed_zero=missed["share with PageValues = 0"],
        caught_zero=caught["share with PageValues = 0"],
        missed_prob=missed["probability"],
        missed_time=missed["median ProductRelated_Duration"],
        caught_time=caught["median ProductRelated_Duration"],
        missed_exit=missed["ExitRates"],
        caught_exit=caught["ExitRates"],
        visitor_sentence=visitor_sentence,
        missed_month=missed_month,
        missed_month_share=month_shares.loc[MISSED_BUYER, missed_month],
        missed_month_caught=month_shares.loc[CAUGHT_BUYER, missed_month],
        fa_pv=false_alarm["PageValues"],
        caught_pv=caught["PageValues"],
        fa_zero=false_alarm["share with PageValues = 0"],
        fa_month=false_alarm_month,
        fa_month_share=month_shares.loc[FALSE_ALARM, false_alarm_month],
        fa_month_caught=month_shares.loc[CAUGHT_BUYER, false_alarm_month],
        fa_returning=visitor_shares.loc[FALSE_ALARM].get("Returning_Visitor", 0.0),
        caught_returning=visitor_shares.loc[CAUGHT_BUYER].get("Returning_Visitor", 0.0),
    )

    with open(os.path.join(OUTPUTS_FOLDER, "error_analysis.md"), "w") as file:
        file.write(text)
    print("\n" + text)
    return text


def run_error_analysis(pipeline, model_name, test_features, test_target, threshold):
    """
    Run the whole error analysis for the best model.

    Inputs:
        pipeline: the fitted best model.
        model_name: its name.
        test_features, test_target: the test data.
        threshold: its decision threshold.
    Returns:
        The labelled sessions DataFrame.
    """
    print("\nError analysis of " + model_name + f" at threshold {threshold:.2f}...")
    sessions = label_prediction_groups(pipeline, test_features, test_target, threshold)

    numerical_summary = summarise_numerical_features(sessions)
    print("\nAverage feature values per group:")
    print(
        numerical_summary[
            [
                "sessions",
                "PageValues",
                "share with PageValues = 0",
                "ProductRelated_Duration",
                "ExitRates",
                "BounceRates",
                "probability",
            ]
        ].to_string()
    )

    category_tables = []
    for column in ["VisitorType", "Month", "Weekend"]:
        shares = summarise_categorical_feature(sessions, column)
        shares.columns = [column + " = " + str(value) for value in shares.columns]
        category_tables.append(shares)
    category_summary = pd.concat(category_tables, axis=1)
    category_summary.to_csv(
        os.path.join(OUTPUTS_FOLDER, "error_categorical_summary.csv"), index_label="group"
    )

    plot_error_analysis(sessions, model_name)
    plot_probability_histogram(sessions, threshold, model_name)
    write_error_findings(sessions, numerical_summary, model_name, threshold)
    return sessions


In [ ]:
best_threshold = chosen_thresholds[best_model_name]["best_profit"]
labelled_sessions = run_error_analysis(models[best_model_name], best_model_name,
                                       test_features, test_target, best_threshold)
show("outputs/figures/error_analysis.png")
show("outputs/figures/error_probability_histogram.png", width=650)
display(Markdown(open("outputs/error_analysis.md").read()))

## 14. Conclusions

- **PageValues is the deciding signal.** Every model ranks it first, the Decision Tree splits on it at the root, and removing it costs every model a large share of its F1.
- **The Decision Tree is the best main model.** It comes close to the XGBoost benchmark while staying fully readable, which matters when a shop team has to trust the rules.
- **XGBoost is the strongest benchmark**, a little ahead on F1 and ROC-AUC, at the price of interpretability.
- **KNN and Naive Bayes do not fit this data well.** KNN suffers from many one-hot dimensions and skewed features (undersampling helps); Naive Bayes' independence assumption breaks and its probabilities are unusable.
- **The gap to XGBoost is small but real.** The confidence intervals overlap, yet McNemar's test (p = 0.004) shows XGBoost makes significantly fewer mistakes on the same sessions.
- **PageValues may leak part of the answer.** The scores without it are the honest lower bound for a live system that only knows page values from earlier periods.
- **The threshold is a business decision.** With a profit model, the best cutoff differs from 0.5 and from the best-F1 cutoff, and it moves with the assumptions; calibrated probabilities make it explainable (p > cost / (profit x boost)) and earned the most profit.
- **Missed buyers are mostly sessions with PageValues = 0**, which look like ordinary browsing; false alarms are mostly returning visitors who reached valuable pages and left. Fixing them needs more data about the visitor, not more tuning.

> **Full project on GitHub:** [https://github.com/pxnz3r/Predator](https://github.com/pxnz3r/Predator) (the `predator` package, tests, documentation and a **Streamlit demo app**). This notebook is a self-contained version: all the code is included below.